# Vaani-Lite SED v3 (final)

**All known bugs fixed:**

1. BEATs cache global-index invariant enforced end to end.
2. LR warmup stepped per optimizer update (not per epoch, not per raw batch).
3. **Short clips:** BEATs padded to canonical 8s length instead of being stretched by `F.interpolate`; fusion gated on `beats_valid_ratio`.
4. **Patch-time mapping:** uses BEATs' fixed patch hop (0.16s) and offset (0.075s), not `duration / T_full`.
5. **Exhaustive cache validation** replaces sampled coverage check; missing cache raises instead of silently falling back to live BEATs.


In [1]:
!nvidia-smi

Wed Sep 16 12:15:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 596.51                 Driver Version: 596.51         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 2000 Blac...  WDDM  |   00000000:02:00.0 Off |                  Off |
| 30%   32C    P8              5W /   70W |       0MiB /  16311MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_capability(0))
x = torch.randn(4, 4, device="cuda") @ torch.randn(4, 4, device="cuda")
print(x.sum().item())  # should just work, no warning

2.14.0+cu132 13.2 (12, 0)
-4.389796257019043


In [3]:
!uname -a

'uname' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
# =============================================================================
# CELL: Setup
# =============================================================================
!rm -rf /kaggle/working/unilm
!git clone --depth 1 https://github.com/microsoft/unilm.git /kaggle/working/unilm
!pip install -q scipy scikit-learn soundfile

import os, sys, random, json, re, csv, io, difflib
from collections import Counter, defaultdict
from copy import deepcopy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
from sklearn.model_selection import GroupShuffleSplit
from scipy.signal import medfilt
import soundfile as sf
from datasets import load_from_disk, disable_caching

disable_caching()

BEATS_DIR = "/kaggle/working/unilm/beats"
if not os.path.isdir(BEATS_DIR):
    raise FileNotFoundError(f"BEATs source not found: {BEATS_DIR}")
sys.path.insert(0, BEATS_DIR)
from BEATs import BEATs, BEATsConfig

DATASET_PATH = (
    "/kaggle/input/datasets/ayanokoji2028/"
    "vaani-noise-hf/vaani_noise_dataset/train"
)
BEATS_CHECKPOINT = (
    "BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt2.pt"
)
# BEATs cache: pure /kaggle/working, no packaged Kaggle Dataset involved.
# beats-cache-full-3 (a prior packaged upload) was confirmed 100% all-zero
# (every cached array read back as exact 0.0 -- almost certainly the old
# fp16-autocast NaN bug getting silently laundered into zeros by
# nan_to_num during a build that predates the fp32 fix). Rather than
# re-publish a Dataset, this rebuilds directly into the working directory
# each session and trains straight from that. KAGGLE_DATASET_CACHE_DIR is
# kept as a hook (None = unused) in case you want a packaged cache again
# later -- every downstream cell already reads BEATS_CACHE_DIRS as a list
# and degrades to a single directory when this is None, so nothing else
# needs to change if you do.
KAGGLE_DATASET_CACHE_DIR = None
BEATS_CACHE_BUILD_DIR = "/kaggle/working/beats_cache_full"      # writable -- the only cache tier now
BEATS_CACHE_DIRS = ([KAGGLE_DATASET_CACHE_DIR] if KAGGLE_DATASET_CACHE_DIR else []) + [BEATS_CACHE_BUILD_DIR]
BEATS_CACHE_DIR = BEATS_CACHE_BUILD_DIR  # kept as an alias for any old single-dir usage

print("BEATs source:", BEATS_DIR)
print("Cache (read, priority order):", BEATS_CACHE_DIRS)
print("Cache (build/writable):", BEATS_CACHE_BUILD_DIR)
if KAGGLE_DATASET_CACHE_DIR and os.path.isdir(KAGGLE_DATASET_CACHE_DIR):
    _n_packaged = len([f for f in os.listdir(KAGGLE_DATASET_CACHE_DIR) if f.endswith(".npy")])
    print(f"  packaged dataset cache: {_n_packaged} .npy files found at {KAGGLE_DATASET_CACHE_DIR}")
else:
    print("  no packaged dataset configured -- reading/writing BEATS_CACHE_BUILD_DIR only")

# Same pure-working-directory approach for the waveform cache (raw-audio
# decode+resample cache, separate from the BEATs feature cache above) --
# no zip/dataset publishing, just build once per session into
# WAV_CACHE_BUILD_DIR and read straight from it. KAGGLE_DATASET_WAV_CACHE_DIR
# is kept as an unused hook (None) for the same reason as its BEATs
# counterpart above.
KAGGLE_DATASET_WAV_CACHE_DIR = None
WAV_CACHE_BUILD_DIR = "/kaggle/working/wav_cache_16k_flac"
WAV_CACHE_DIR = WAV_CACHE_BUILD_DIR  # alias, mirrors BEATS_CACHE_DIR
WAV_CACHE_DIRS = ([KAGGLE_DATASET_WAV_CACHE_DIR] if KAGGLE_DATASET_WAV_CACHE_DIR else []) + [WAV_CACHE_BUILD_DIR]
print("Waveform cache (read, priority order):", WAV_CACHE_DIRS)
if KAGGLE_DATASET_WAV_CACHE_DIR and os.path.isdir(KAGGLE_DATASET_WAV_CACHE_DIR):
    _n_wav_packaged = len([f for f in os.listdir(KAGGLE_DATASET_WAV_CACHE_DIR) if f.endswith(".flac")])
    print(f"  packaged waveform cache: {_n_wav_packaged} .flac files found at {KAGGLE_DATASET_WAV_CACHE_DIR}")
else:
    print("  no packaged waveform dataset configured -- reading/writing WAV_CACHE_BUILD_DIR only")

# BEATs fbank framing is 25 ms window / 10 ms hop, and patch_embedding has
# stride = input_patch_size = 16. So each BEATs patch advances 16 * 10 ms
# = 160 ms, with the first patch centered ~75 ms into the clip. These constants
# replace the previous `duration / T_full` approximation.
BEATS_FBANK_HOP_SEC    = 0.010
BEATS_PATCH_SIZE       = 16
BEATS_PATCH_HOP_SEC    = BEATS_FBANK_HOP_SEC * BEATS_PATCH_SIZE          # 0.16 s
BEATS_PATCH_OFFSET_SEC = (BEATS_PATCH_SIZE - 1) * BEATS_FBANK_HOP_SEC / 2  # 0.075 s
print(f"BEATs patch hop={BEATS_PATCH_HOP_SEC}s offset={BEATS_PATCH_OFFSET_SEC}s")

In [2]:
# =============================================================================
# CELL: Tag canonicalization (unchanged)
# =============================================================================
_BRACKET_RE = re.compile(r"^[\[]?<(.+)>[\]]?$")
_SEED_SYNONYMS = {
    "crow cawaing": "crow cawing", "lip_smacking": "lip smacking",
    "cawaing": "cawing", "yawing": "yawning", "sniffling": "sniffing",
    "hiccups": "hiccup", "nose sneezing": "sneezing",
    "nose clearing": "throat clearing", "cleaning throat": "throat clearing",
    "throat cleaning": "throat clearing",
}
_INTERCHANGEABLE_SUFFIXES = {"noise", "sound", "voice", "playing", "tone", "song"}
_PLURAL_MAP = {"children": "child", "kids": "kid", "birds": "bird", "bugs": "bug",
               "insects": "insect", "vehicles": "vehicle", "songs": "song", "crickets": "cricket"}

def normalize_tag(raw_tag):
    m = _BRACKET_RE.match(raw_tag.strip())
    inner = m.group(1) if m else raw_tag.strip()
    inner = inner.replace("_", " ").lower().strip()
    inner = re.sub(r"\s+", " ", inner)
    return _SEED_SYNONYMS.get(inner, inner)

def _canonical_tokens(norm_tag):
    tokens = [_PLURAL_MAP.get(t, t) for t in norm_tag.split(" ")]
    core = [t for t in tokens if t not in _INTERCHANGEABLE_SUFFIXES]
    return tuple(core) if core else tuple(tokens)

def build_tag_clusters(tag_counts, similarity_threshold=0.82):
    normalized = {raw: normalize_tag(raw) for raw in tag_counts}
    token_key_to_raws = defaultdict(list)
    for raw, norm in normalized.items():
        token_key_to_raws[_canonical_tokens(norm)].append(raw)
    clusters = list(token_key_to_raws.values())
    keys = list(token_key_to_raws.keys())
    merged, used = [], [False] * len(clusters)
    for i in range(len(clusters)):
        if used[i]: continue
        group = list(clusters[i]); used[i] = True
        key_i = " ".join(keys[i])
        for j in range(i + 1, len(clusters)):
            if used[j]: continue
            if difflib.SequenceMatcher(None, key_i, " ".join(keys[j])).ratio() >= similarity_threshold:
                group.extend(clusters[j]); used[j] = True
        merged.append(group)
    raw_to_canonical, cluster_report = {}, []
    for group in merged:
        rep = max(group, key=lambda r: tag_counts.get(r, 0))
        canonical = normalize_tag(rep)
        total = sum(tag_counts.get(r, 0) for r in group)
        for r in group: raw_to_canonical[r] = canonical
        cluster_report.append({"canonical_tag": canonical, "total_count": total,
                               "n_raw_variants": len(group),
                               "raw_variants": sorted(group, key=lambda r: -tag_counts.get(r, 0))})
    cluster_report.sort(key=lambda r: -r["total_count"])
    return raw_to_canonical, cluster_report

def write_cluster_report_csv(cluster_report, path):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["canonical_tag", "total_count", "n_raw_variants", "raw_variants"])
        for row in cluster_report:
            w.writerow([row["canonical_tag"], row["total_count"], row["n_raw_variants"],
                        " | ".join(row["raw_variants"])])

In [3]:
# =============================================================================
# CELL: Constants and helpers
# =============================================================================
COARSE_CATEGORIES = ["animal", "appliance_machine", "baby_child", "human_non_speech",
                     "phone_signal_alarm", "singing_music", "vehicle_traffic"]
COARSE_TO_IDX = {c: i for i, c in enumerate(COARSE_CATEGORIES)}

TIER_VERIFIED, TIER_UNVERIFIED, TIER_WEAK_ONLY, TIER_PSEUDO_BRONZE = 0, 1, 2, 3
_QUALITY_TO_TIER = {"verified_timestamps": TIER_VERIFIED,
                    "unverified_timestamps": TIER_UNVERIFIED,
                    "no_timestamps": TIER_WEAK_ONLY}

def split_indices_by_tier(dataset, indices=None):
    """Returns tier -> list of GLOBAL indices (positions into `dataset`)."""
    quality_col = dataset["annotationQuality"]
    idxs = range(len(dataset)) if indices is None else list(indices)
    gold, silver, bronze = [], [], []
    for i in idxs:
        q = quality_col[i]
        if q == "verified_timestamps": gold.append(i)
        elif q == "unverified_timestamps": silver.append(i)
        elif q == "no_timestamps": bronze.append(i)
    return {"gold": gold, "silver": silver, "bronze": bronze}

def _load_audio(audio_field):
    if audio_field.get("array") is not None:
        wav = np.asarray(audio_field["array"], dtype=np.float32)
        return wav, int(audio_field["sampling_rate"])
    if audio_field.get("bytes") is not None:
        wav, sr = sf.read(io.BytesIO(audio_field["bytes"]), dtype="float32")
    elif audio_field.get("path"):
        wav, sr = sf.read(audio_field["path"], dtype="float32")
    else:
        raise ValueError(f"Unknown audio field: {list(audio_field.keys())}")
    if wav.ndim > 1: wav = wav.mean(axis=1)
    return wav.astype(np.float32), int(sr)

def resolve_wav_cache_path(full_idx, cache_dir_or_dirs):
    """Same lookup pattern as resolve_cache_path (checks each dir in
    priority order, first match wins), but for the FLAC waveform cache --
    kept as a separate function rather than overloading resolve_cache_path
    so every existing BEATs-cache call site (.npy) is untouched."""
    for d in _normalize_cache_dirs(cache_dir_or_dirs):
        p = os.path.join(d, f"{int(full_idx):06d}.flac")
        if os.path.exists(p):
            return p
    return None

def _load_waveform(full_idx, parent_hf, sample_rate, wav_cache_dir=None):
    """Loads a resampled mono waveform for full_idx. Uses the waveform
    cache when available (no audio decode, no row access at all); falls
    back to a live decode+resample otherwise. wav_cache_dir may be a
    single directory (old behavior) or a list -- read priority order,
    mirroring BEATS_CACHE_DIRS: e.g. a read-only packaged Kaggle Dataset
    checked first, the writable working-dir cache second. Self-healing
    writes always go to the LAST directory in the list, by convention the
    writable build dir (any earlier ones are read-only packaged data)."""
    dirs = _normalize_cache_dirs(wav_cache_dir)
    if dirs:
        p = resolve_wav_cache_path(full_idx, dirs)
        if p is not None:
            wav_np, _ = sf.read(p, dtype="float32")
            return torch.from_numpy(np.asarray(wav_np, dtype=np.float32))
    wav_np, sr = _load_audio(parent_hf[int(full_idx)]["audio"])
    wav = torch.from_numpy(np.asarray(wav_np, dtype=np.float32))
    if wav.ndim > 1: wav = wav.mean(dim=-1)
    if sr != sample_rate:
        wav = torchaudio.functional.resample(wav, int(sr), sample_rate)
    wav = wav.contiguous()
    if dirs:
        write_dir = dirs[-1]
        try:
            os.makedirs(write_dir, exist_ok=True)
            sf.write(os.path.join(write_dir, f"{int(full_idx):06d}.flac"),
                     wav.numpy(), sample_rate, format="FLAC", subtype="PCM_16")
        except Exception:
            pass
    return wav

def build_label_table(hf_dataset):
    """Pulls ONLY the columns needed for labels via columnar access, once.
    This is what actually makes the waveform cache pay off: HF datasets
    decode the audio column as a side effect of ANY row access
    (hf_dataset[i]), even when you never touch that field -- so without
    this, every __getitem__ would still fully decode audio just to read
    NoiseCategory/NoiseSubCategoryTimeStamp/annotationQuality off the same
    row, then discard the decoded audio in favor of the cached waveform.
    Column access (hf_dataset["col"]) does not trigger that decode."""
    cols = hf_dataset.column_names
    return {
        "NoiseCategory": hf_dataset["NoiseCategory"],
        "NoiseSubCategoryTimeStamp": hf_dataset["NoiseSubCategoryTimeStamp"],
        "annotationQuality": hf_dataset["annotationQuality"],
        "id": hf_dataset["id"] if "id" in cols else [None] * len(hf_dataset),
    }

def _ref_events_for_full_idx(full_idx, label_table):
    evs = []
    for e in (label_table["NoiseSubCategoryTimeStamp"][full_idx] or []):
        try: s = float(e["start"]); en = float(e["end"])
        except (TypeError, ValueError, KeyError): continue
        if en > s: evs.append((s, en))
    return evs

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def _normalize_cache_dirs(cache_dir_or_dirs):
    if cache_dir_or_dirs is None: return []
    if isinstance(cache_dir_or_dirs, (list, tuple)): return list(cache_dir_or_dirs)
    return [cache_dir_or_dirs]

def resolve_cache_path(full_idx, cache_dir_or_dirs):
    """Checks each directory in priority order (e.g. a read-only packaged
    Kaggle Dataset first, the writable working-dir cache second) and
    returns the first existing path, or None if not found in any of them.
    Accepts either a single directory string (old behavior, unchanged) or
    a list of directories (new two-tier behavior)."""
    for d in _normalize_cache_dirs(cache_dir_or_dirs):
        p = os.path.join(d, f"{int(full_idx):06d}.npy")
        if os.path.exists(p):
            return p
    return None

def _validate_cache_exhaustive(all_required_indices, cache_dir, label=""):
    """Verify EVERY required index has a cache file in AT LEAST ONE of the
    given directories (str or list). Raises if any are missing everywhere.
    This is the only acceptable pre-training check -- sampling leaves room
    for thousands of silent mismatches."""
    all_required_indices = sorted({int(i) for i in all_required_indices})
    missing = [fi for fi in all_required_indices if resolve_cache_path(fi, cache_dir) is None]
    tag = f"[{label}] " if label else ""
    if missing:
        raise RuntimeError(
            f"{tag}BEATs cache incomplete: {len(missing)} of "
            f"{len(all_required_indices)} required indices missing from "
            f"{_normalize_cache_dirs(cache_dir)}. First 10: {missing[:10]}. "
            f"Rebuild the cache before training -- the notebook does not "
            f"fall back to live BEATs.")
    print(f"  {tag}[cache-validate] all {len(all_required_indices)} required "
          f"indices present across {_normalize_cache_dirs(cache_dir)}")
    return True

In [4]:
# =============================================================================
# CELL: Mel frontend (unchanged)
# =============================================================================
class MelFrontend(nn.Module):
    def __init__(self, n_mels=128, sample_rate=16000, n_fft=2048,
                 hop_length=160, max_seconds=8.0):
        super().__init__()
        self.sample_rate = int(sample_rate); self.hop_length = int(hop_length)
        self.max_seconds = float(max_seconds); self.n_mels = int(n_mels)
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=self.sample_rate, n_fft=n_fft, hop_length=self.hop_length,
            n_mels=self.n_mels, power=2.0)
        self.to_db = torchaudio.transforms.AmplitudeToDB(stype="power")
    def forward(self, waveform):
        if waveform.ndim == 3: waveform = waveform.squeeze(1)
        max_len = int(round(self.max_seconds * self.sample_rate))
        if waveform.shape[1] > max_len: waveform = waveform[:, :max_len]
        elif waveform.shape[1] < max_len: waveform = F.pad(waveform, (0, max_len - waveform.shape[1]))
        mel = self.to_db(self.mel_spec(waveform))
        mean = mel.mean(dim=(1, 2), keepdim=True)
        std = mel.std(dim=(1, 2), keepdim=True).clamp_min(1e-5)
        return (mel - mean) / std

def apply_mel_specaugment(mel, time_masks=2, freq_masks=2, max_time_frac=0.08, max_freq_bins=12):
    if mel is None: return None
    out = mel.clone(); B, Freq, Time = out.shape
    max_t = max(1, int(Time * max_time_frac))
    for b in range(B):
        for _ in range(time_masks):
            w = random.randint(0, max_t)
            if w:
                s = random.randint(0, max(0, Time - w)); out[b, :, s:s+w] = 0
        for _ in range(freq_masks):
            w = random.randint(0, min(max_freq_bins, Freq))
            if w:
                s = random.randint(0, max(0, Freq - w)); out[b, s:s+w, :] = 0
    return out

In [5]:
# =============================================================================
# CELL: DFD-LKA encoder
# =============================================================================
class DFDConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, n_kernels=4,
                 dilations=(1, 2, 3, 4)):
        super().__init__()
        self.kernel_size = kernel_size
        self.dilations = list(dilations)[:n_kernels]
        self.kernels = nn.ParameterList([
            nn.Parameter(torch.randn(out_ch, in_ch, kernel_size, kernel_size) * 0.02)
            for _ in self.dilations])
        self.biases = nn.ParameterList([nn.Parameter(torch.zeros(out_ch)) for _ in self.dilations])
        self.attn = nn.Sequential(nn.AdaptiveAvgPool2d((None, 1)),
                                  nn.Conv2d(in_ch, n_kernels, kernel_size=1))
    def forward(self, x):
        logits = self.attn(x).squeeze(-1)
        weights = F.softmax(logits, dim=1)
        H, W = x.shape[-2], x.shape[-1]
        out = 0.0
        for k, (kernel, bias, d) in enumerate(zip(self.kernels, self.biases, self.dilations)):
            pad = (self.kernel_size - 1) * d // 2
            y = F.conv2d(x, kernel, bias, padding=pad, dilation=d)
            if y.shape[-2] != H or y.shape[-1] != W: y = y[..., :H, :W]
            w = weights[:, k, :].unsqueeze(1).unsqueeze(-1)
            out = out + y * w
        return out

class LKAAttention(nn.Module):
    def __init__(self, channels, kernel_size=7, dilation=1):
        super().__init__()
        pad = (kernel_size // 2) * dilation
        self.dw = nn.Conv2d(channels, channels, kernel_size, padding=pad,
                            dilation=dilation, groups=channels)
        self.pw = nn.Conv2d(channels, channels, 1)
        self.gate = nn.Sigmoid()
    def forward(self, x):
        return x * self.gate(self.pw(self.dw(x))) + x

class DFDLKABlock(nn.Module):
    def __init__(self, in_ch, out_ch, n_kernels=4, lka_kernel=7):
        super().__init__()
        self.dfd = DFDConv2d(in_ch, out_ch, n_kernels=n_kernels)
        self.bn = nn.BatchNorm2d(out_ch); self.act = nn.GELU()
        self.lka = LKAAttention(out_ch, kernel_size=lka_kernel)
    def forward(self, x):
        return self.lka(self.act(self.bn(self.dfd(x))))

class DFDLKAEncoder(nn.Module):
    def __init__(self, in_ch=1, channels=(32, 64, 128, 192),
                 freq_reduction=(2, 2, 2, 1), n_kernels=4, lka_kernel=7):
        super().__init__()
        self.blocks = nn.ModuleList(); self.pools = nn.ModuleList()
        prev = in_ch
        for ch, fr in zip(channels, freq_reduction):
            self.blocks.append(DFDLKABlock(prev, ch, n_kernels, lka_kernel))
            self.pools.append(nn.MaxPool2d((fr, 1)) if fr > 1 else nn.Identity())
            prev = ch
        self.final_freq_pool = nn.AdaptiveAvgPool2d((1, None))
    def forward(self, x):
        for block, pool in zip(self.blocks, self.pools): x = pool(block(x))
        x = self.final_freq_pool(x).squeeze(2)
        return x.transpose(1, 2)

class TemporalRefinement(nn.Module):
    def __init__(self, dim, kernel_size=5):
        super().__init__()
        self.dw = nn.Conv1d(dim, dim, kernel_size, padding=kernel_size//2, groups=dim)
        self.norm = nn.LayerNorm(dim); self.act = nn.GELU()
    def forward(self, x):
        y = self.dw(x.transpose(1, 2)).transpose(1, 2)
        return self.act(self.norm(x + y))

In [6]:
# =============================================================================
# CELL: BEATs temporal extractor + loader + cache builder
# =============================================================================
def beats_extract_temporal(beats_model, source, padding_mask=None,
                           fbank_mean=15.41663, fbank_std=6.55582):
    fbank = beats_model.preprocess(source, fbank_mean=fbank_mean, fbank_std=fbank_std)
    if padding_mask is not None:
        padding_mask = beats_model.forward_padding_mask(fbank, padding_mask)
    fbank = fbank.unsqueeze(1)
    feat2d = beats_model.patch_embedding(fbank)
    T_patches, F_patches = feat2d.shape[2], feat2d.shape[3]
    feat_seq = feat2d.reshape(feat2d.shape[0], feat2d.shape[1], -1).transpose(1, 2)
    feat_seq = beats_model.layer_norm(feat_seq)
    if padding_mask is not None:
        padding_mask = beats_model.forward_padding_mask(feat_seq, padding_mask)
    if beats_model.post_extract_proj is not None:
        feat_seq = beats_model.post_extract_proj(feat_seq)
    x = beats_model.dropout_input(feat_seq)
    x, _ = beats_model.encoder(x, padding_mask=padding_mask)
    x = x.reshape(x.shape[0], T_patches, F_patches, x.shape[-1])
    x_time = x.mean(dim=2)
    return x_time, T_patches, None

def load_beats_model(checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    beats_cfg = BEATsConfig(checkpoint["cfg"])
    print("BEATs cfg: layers=", beats_cfg.encoder_layers,
          " dim=", beats_cfg.encoder_embed_dim,
          " patch=", getattr(beats_cfg, "input_patch_size", "n/a"))
    beats_model = BEATs(beats_cfg)
    beats_model.load_state_dict(checkpoint["model"], strict=True)
    beats_model.predictor = None
    beats_model.to(device).eval()
    for p in beats_model.parameters(): p.requires_grad_(False)
    print("BEATs loaded and frozen.")
    return beats_model, beats_cfg.encoder_embed_dim

@torch.no_grad()
def extract_beats_full(beats_model, waveform_1d, device):
    wav = waveform_1d.to(device).unsqueeze(0)
    # FIXED: was enabled=(device.type == "cuda"). Running a real, deep,
    # pretrained transformer under fp16 autocast produced all-NaN output
    # here (confirmed empirically -- fp16 gave NaN=True, fp32 gave a clean
    # min/max of -1.58/2.29 on the same clip). BEATs is frozen and this
    # only runs once per clip during caching, so there is no speed reason
    # to keep it in fp16 -- autocast is for things you backprop through.
    with autocast(device_type="cuda", enabled=False):
        feat, T_patches, _ = beats_extract_temporal(beats_model, wav)
    return feat[0].float().cpu().numpy(), T_patches

def build_beats_cache(hf_dataset, beats_model, device, cache_dir, sample_rate=16000,
                      max_clips=None, log_every=200, also_check_dirs=None):
    """Cache keyed by FULL-dataset index (i in range(len(hf_dataset))).
    Always WRITES to cache_dir (must be writable). also_check_dirs, if
    given, is a read-only-dirs-first check -- an index already present
    there (e.g. in a packaged Kaggle Dataset) is skipped entirely, so this
    never wastes time recomputing something you already have."""
    os.makedirs(cache_dir, exist_ok=True)
    n = len(hf_dataset) if max_clips is None else min(max_clips, len(hf_dataset))
    read_dirs = _normalize_cache_dirs(also_check_dirs) + [cache_dir]
    manifest = {}
    n_skipped_from_other_dirs = 0
    for i in range(n):
        path = os.path.join(cache_dir, f"{i:06d}.npy")
        if resolve_cache_path(i, read_dirs) is not None:
            if not os.path.exists(path):
                n_skipped_from_other_dirs += 1
            continue
        try:
            wav_np, sr = _load_audio(hf_dataset[i]["audio"])
            wav = torch.from_numpy(np.asarray(wav_np, dtype=np.float32))
            if wav.ndim > 1: wav = wav.mean(dim=-1)
            if sr != sample_rate:
                wav = torchaudio.functional.resample(wav, sr, sample_rate)
            feat, T_patches = extract_beats_full(beats_model, wav, device)

            # NEW: loud, unmissable warning if raw BEATs output is ever bad
            # again -- this is what should have existed the first time
            # instead of letting nan_to_num silently launder NaN into
            # deceptively clean-looking zeros with no trace left behind.
            if not np.isfinite(feat).all():
                print(f"  [cache] WARNING idx={i}: raw BEATs output has NaN/Inf before "
                      f"sanitizing ({np.isnan(feat).mean()*100:.1f}% NaN, "
                      f"{np.isinf(feat).mean()*100:.1f}% Inf)")

            # Kept as a backstop for genuinely corrupt individual clips only
            # -- with the fp32 fix above, this should now rarely if ever
            # actually trigger.
            feat32 = np.nan_to_num(feat.astype(np.float32), nan=0.0, posinf=6.0e4, neginf=-6.0e4)
            feat16 = np.clip(feat32, -6.0e4, 6.0e4).astype(np.float16)
            np.save(path, feat16)
            manifest[i] = {"T_patches": int(T_patches),
                           "duration": float(wav.numel() / sample_rate)}
        except Exception as e:
            print(f"  [cache] failed idx={i}: {e}")
        if (i + 1) % log_every == 0: print(f"  [cache] {i+1}/{n}")
    manifest_path = os.path.join(cache_dir, "manifest.json")
    existing = {}
    if os.path.exists(manifest_path):
        with open(manifest_path) as f: existing = json.load(f)
    existing.update({str(k): v for k, v in manifest.items()})
    with open(manifest_path, "w") as f: json.dump(existing, f)
    print(f"[cache] done. newly built here: {len(existing)}, "
          f"skipped ({n_skipped_from_other_dirs} already in another configured dir, "
          f"rest already in {cache_dir})")
    return manifest

# =============================================================================
# Waveform cache -- decode+resample once, reuse every epoch.
# Stored as FLAC (lossless, ~40-60% the size of raw float16 PCM) rather
# than .npy: the raw-array cache filled /kaggle/working's disk quota
# mid-build once already (16kHz float16 mono is 32KB/sec uncompressed --
# roughly 23GB across the full dataset), and FLAC keeps the same speed
# win (still skip decode+resample every epoch) at roughly half the disk
# footprint. Mirrors build_beats_cache's two-tier pattern exactly: skips
# indices already present in any read-only packaged dir (also_check_dirs),
# only writes to the given writable cache_dir. Unlike the BEATs cache, a
# missing entry here falls back gracefully to a live decode (see
# _load_waveform) rather than raising -- this cache is purely a speed
# optimization, not a correctness dependency.
# =============================================================================
def build_waveform_cache(hf_dataset, indices, cache_dir, sample_rate=16000,
                         log_every=2000, also_check_dirs=None):
    os.makedirs(cache_dir, exist_ok=True)
    indices = sorted({int(i) for i in indices})
    read_dirs = _normalize_cache_dirs(also_check_dirs) + [cache_dir]
    todo = [i for i in indices if resolve_wav_cache_path(i, read_dirs) is None]
    n_skipped = len(indices) - len(todo)
    print(f"  [wav-cache] {len(todo)} of {len(indices)} indices need caching "
          f"({n_skipped} already present in {read_dirs})")
    n_failed = 0
    for n, i in enumerate(todo):
        try:
            wav_np, sr = _load_audio(hf_dataset[i]["audio"])
            wav = torch.from_numpy(np.asarray(wav_np, dtype=np.float32))
            if wav.ndim > 1: wav = wav.mean(dim=-1)
            if sr != sample_rate:
                wav = torchaudio.functional.resample(wav, int(sr), sample_rate)
            sf.write(os.path.join(cache_dir, f"{i:06d}.flac"),
                     wav.numpy(), sample_rate, format="FLAC", subtype="PCM_16")
        except Exception as e:
            n_failed += 1
            print(f"  [wav-cache] failed idx={i}: {e}")
        if (n + 1) % log_every == 0:
            print(f"    [wav-cache] {n + 1}/{len(todo)} cached ({n_failed} failed so far)")
    print(f"  [wav-cache] done: {cache_dir} ({n_failed} failed, {n_skipped} skipped)")


In [7]:
# =============================================================================
# CELL: VaaniNoiseDataset + collate_fn
#
# - Global-index invariant: cache + HF row indexed by same integer.
# - Short-clip BEATs padded to canonical 8s length; per-sample valid ratio
#   propagates so fusion can gate padded frames.
# - Patch-time slicing uses BEATs' fixed hop/offset, not duration/T_full.
# - STRICT lookup: missing cache raises, never silent zero-fill.
# =============================================================================
class VaaniNoiseDataset(Dataset):
    def __init__(self, parent_hf, full_indices, raw_to_canonical, canonical_tags,
                 fine_to_coarse_idx, beats_cache_dir=None, wav_cache_dir=None,
                 label_table=None, pseudo_overrides=None, crop_seconds=8.0, sample_rate=16000,
                 training=True, background_crop_prob=0.3, event_jitter_seconds=1.5):
        self.parent = parent_hf
        self.full_indices = np.asarray(full_indices, dtype=np.int64)
        assert self.full_indices.ndim == 1 and len(self.full_indices) > 0
        assert int(self.full_indices.max()) < len(parent_hf), \
            f"full_indices out of range: max={int(self.full_indices.max())} parent={len(parent_hf)}"
        self.raw_to_canonical = raw_to_canonical
        self.canonical_tags = canonical_tags
        self.tag_to_idx = {t: i for i, t in enumerate(canonical_tags)}
        self.fine_to_coarse_idx = fine_to_coarse_idx
        self.pseudo_overrides = pseudo_overrides or {}
        self.crop_seconds = float(crop_seconds); self.sample_rate = int(sample_rate)
        self.crop_samples = int(round(self.crop_seconds * self.sample_rate)) if self.crop_seconds > 0 else 0
        self.training = bool(training)
        self.background_crop_prob = float(background_crop_prob)
        self.event_jitter_seconds = float(event_jitter_seconds)
        self.beats_cache_dir = beats_cache_dir
        # NEW: wav_cache_dir avoids re-decoding audio every epoch;
        # label_table avoids the row-access that would otherwise decode
        # audio anyway just to read labels off the same row. Both optional
        # and independently gracefully-degrading -- omit either and this
        # dataset behaves exactly as before, just slower.
        self.wav_cache_dir = wav_cache_dir
        self.label_table = label_table
        self.canonical_T = (max(1, int(round(self.crop_seconds / BEATS_PATCH_HOP_SEC)))
                            if self.crop_seconds > 0 else 1)
        self._debug_prints_left = 3

    def __len__(self):
        return len(self.full_indices)

    def _full_idx(self, local_idx):
        return int(self.full_indices[local_idx])

    def _cache_path(self, full_idx):
        """Display/debug only -- returns the resolved path if found in any
        configured cache dir, else the join against the first dir (so a
        MISSING-file print still shows a sensible path, not None)."""
        found = resolve_cache_path(full_idx, self.beats_cache_dir)
        if found is not None:
            return found
        first_dir = _normalize_cache_dirs(self.beats_cache_dir)
        return os.path.join(first_dir[0], f"{int(full_idx):06d}.npy") if first_dir else "n/a"

    def _fine_idx(self, raw_tag):
        canon = self.raw_to_canonical.get(raw_tag, normalize_tag(raw_tag))
        return self.tag_to_idx.get(canon)

    def _resample_mono(self, wav_np, sr):
        wav = torch.from_numpy(np.asarray(wav_np, dtype=np.float32))
        if wav.ndim > 1: wav = wav.mean(dim=-1)
        if sr != self.sample_rate:
            wav = torchaudio.functional.resample(wav, int(sr), self.sample_rate)
        return wav.contiguous()

    def _load_beats_full(self, full_idx):
        if self.beats_cache_dir is None:
            raise RuntimeError("beats_cache_dir is required for this dataset.")
        path = resolve_cache_path(full_idx, self.beats_cache_dir)
        if path is None:
            raise RuntimeError(
                f"Missing BEATs cache entry for full_idx={full_idx} in "
                f"{_normalize_cache_dirs(self.beats_cache_dir)}. "
                f"Run build_beats_cache before training.")
        return np.load(path).astype(np.float32)

    def _slice_beats_crop(self, beats_full, start_s, end_s):
        """Patch-time slicing using BEATs' fixed hop/offset."""
        T_full = beats_full.shape[0]
        p0 = int(np.floor((start_s - BEATS_PATCH_OFFSET_SEC) / BEATS_PATCH_HOP_SEC))
        p1 = int(np.ceil((end_s   - BEATS_PATCH_OFFSET_SEC) / BEATS_PATCH_HOP_SEC)) + 1
        p0 = max(0, min(p0, T_full - 1))
        p1 = max(p0 + 1, min(p1, T_full))
        return beats_full[p0:p1]

    def _pad_or_truncate_to_canonical(self, bw):
        if bw is None: return None
        if bw.shape[0] < self.canonical_T:
            pad = np.zeros((self.canonical_T - bw.shape[0], bw.shape[1]), dtype=bw.dtype)
            return np.concatenate([bw, pad], axis=0)
        if bw.shape[0] > self.canonical_T:
            return bw[:self.canonical_T]
        return bw

    def _select_crop(self, wav, event_spans, beats_full):
        n = wav.numel(); full_duration = n / self.sample_rate
        if self.crop_seconds <= 0:
            return wav, event_spans, full_duration, beats_full, 1.0
        target = self.crop_samples

        # -------- Short clip: pad waveform AND pad BEATs to canonical length --
        if n <= target:
            valid_seconds = n / self.sample_rate
            wav_p = F.pad(wav, (0, target - n))
            if beats_full is not None:
                bw = self._pad_or_truncate_to_canonical(beats_full)
                valid_ratio = min(1.0, valid_seconds / self.crop_seconds)
            else:
                bw = None; valid_ratio = 0.0
            return wav_p, event_spans, valid_seconds, bw, valid_ratio

        # -------- Normal crop inside a longer recording ----------------------
        take_bg = (not event_spans) or (random.random() < self.background_crop_prob)
        if take_bg:
            start = random.randint(0, n - target)
        else:
            s0, e0 = random.choice(event_spans)
            center = (s0 + e0) * 0.5 + random.uniform(-self.event_jitter_seconds, self.event_jitter_seconds)
            start = int(round(center * self.sample_rate - target / 2))
            start = max(0, min(start, n - target))
        end = start + target
        crop = wav[start:end]
        cs = start / self.sample_rate; ce = end / self.sample_rate
        cropped = []
        for s, e in event_spans:
            ns = max(0.0, float(s) - cs); ne = min(self.crop_seconds, float(e) - cs)
            if ne > ns: cropped.append((ns, ne))
        if beats_full is not None:
            bw = self._slice_beats_crop(beats_full, cs, ce)
            bw = self._pad_or_truncate_to_canonical(bw)
            valid_ratio = 1.0
        else:
            bw = None; valid_ratio = 0.0
        return crop, cropped, self.crop_seconds, bw, valid_ratio

    def __getitem__(self, idx, _retries=0):
        full_idx = self._full_idx(idx)
        if self._debug_prints_left > 0:
            print(f"    [dataset] local={idx} full_idx={full_idx} "
                  f"cache={self._cache_path(full_idx) if self.beats_cache_dir else 'n/a'}")
            self._debug_prints_left -= 1
        try:
            # CHANGED: was `self.parent[full_idx]` + `_load_audio` +
            # `_resample_mono` every single call. Uses the waveform cache
            # when available (no decode at all); falls back to a live
            # decode only for indices the cache doesn't have yet.
            wav = _load_waveform(full_idx, self.parent, self.sample_rate, self.wav_cache_dir)
        except Exception as exc:
            if _retries >= 5: raise RuntimeError(f"Failed to load audio full_idx={full_idx}") from exc
            return self.__getitem__(random.randrange(len(self)), _retries + 1)

        # CHANGED: was `sample = self.parent[full_idx]` then read fields off
        # it -- that row access decodes the audio column as a side effect
        # regardless of whether it's used, which would have made the
        # waveform cache above pointless. label_table (columnar, built
        # once) avoids that entirely; parent_hf row access is now only a
        # fallback for the rare case label_table wasn't provided.
        if self.label_table is not None:
            cats = self.label_table["NoiseCategory"][full_idx] or []
            raw_events = self.label_table["NoiseSubCategoryTimeStamp"][full_idx] or []
            quality = self.label_table["annotationQuality"][full_idx]
        else:
            sample = self.parent[full_idx]
            cats = sample.get("NoiseCategory") or []
            raw_events = sample.get("NoiseSubCategoryTimeStamp") or []
            quality = sample.get("annotationQuality")

        coarse_target = torch.zeros(len(COARSE_CATEGORIES), dtype=torch.float32)
        for cat in cats:
            if cat in COARSE_TO_IDX: coarse_target[COARSE_TO_IDX[cat]] = 1.0
        tier = _QUALITY_TO_TIER.get(quality, TIER_WEAK_ONLY)
        override = self.pseudo_overrides.get(full_idx)
        fine_clip_target = torch.zeros(len(self.canonical_tags), dtype=torch.float32)
        event_spans = []
        if override is not None:
            tier = TIER_PSEUDO_BRONZE
            for ev in override.get("events", []):
                try: s = float(ev["start"]); e = float(ev["end"])
                except (TypeError, ValueError, KeyError): continue
                if e > s: event_spans.append((s, e))
        else:
            for e in raw_events:
                try: s = float(e["start"]); en = float(e["end"])
                except (TypeError, ValueError, KeyError): continue
                if en <= s: continue
                event_spans.append((s, en))
                fi = self._fine_idx(e.get("tag", ""))
                if fi is not None: fine_clip_target[fi] = 1.0

        beats_full = self._load_beats_full(full_idx)   # STRICT

        if self.training and self.crop_seconds > 0:
            waveform, event_spans, valid_seconds, beats_crop, beats_valid_ratio = \
                self._select_crop(wav, event_spans, beats_full)
        else:
            waveform = wav
            valid_seconds = wav.numel() / self.sample_rate
            beats_crop = self._pad_or_truncate_to_canonical(beats_full)
            beats_valid_ratio = 1.0

        event_clip_target = torch.tensor(
            1.0 if (len(event_spans) > 0 or coarse_target.sum() > 0 or fine_clip_target.sum() > 0) else 0.0,
            dtype=torch.float32)

        return {
            "waveform": waveform.float().unsqueeze(0),
            "beats_crop": torch.from_numpy(np.asarray(beats_crop, dtype=np.float32)),
            "beats_valid_ratio": float(beats_valid_ratio),
            "duration": float(valid_seconds),
            "coarse_target": coarse_target,
            "fine_clip_target": fine_clip_target,
            "event_clip_target": event_clip_target,
            "event_spans": event_spans,
            "tier": int(tier),
            "src_idx": int(idx),
            "full_idx": int(full_idx),
        }

    def verify_cache_alignment(self, n_samples=5):
        if self.beats_cache_dir is None:
            print("  [cache-verify] no cache dir"); return
        k = min(n_samples, len(self))
        check = random.sample(range(len(self)), k) if len(self) > k else list(range(len(self)))
        print(f"  [cache-verify] checking {k}/{len(self)} samples "
              f"(expected canonical_T={self.canonical_T})")
        for li in check:
            fi = self._full_idx(li)
            cp = resolve_cache_path(fi, self.beats_cache_dir)
            if cp is None:
                print(f"    MISSING: local={li} full_idx={fi}"); continue
            feat = np.load(cp, mmap_mode="r")
            wav_np, sr = _load_audio(self.parent[fi]["audio"])
            dur = wav_np.shape[0] / sr
            expected_T = max(1, int(round(dur / BEATS_PATCH_HOP_SEC)))
            ratio = feat.shape[0] / expected_T
            flag = "OK" if 0.85 < ratio < 1.15 else "SUSPECT"
            print(f"    {flag}: local={li} full_idx={fi} dur={dur:.2f}s "
                  f"T_cached={feat.shape[0]} expected~{expected_T} ratio={ratio:.2f}")


def collate_fn(batch):
    waveforms = torch.cat([b["waveform"] for b in batch], dim=0)
    beats_list = [b["beats_crop"] for b in batch]
    T_max = max(f.shape[0] for f in beats_list)
    D = beats_list[0].shape[1]
    beats_padded = torch.zeros(len(batch), T_max, D, dtype=torch.float32)
    beats_mask = torch.zeros(len(batch), T_max, dtype=torch.float32)
    for i, f in enumerate(beats_list):
        beats_padded[i, :f.shape[0]] = f
        beats_mask[i, :f.shape[0]] = 1.0
    return {
        "waveform": waveforms,
        "beats_padded": beats_padded,
        "beats_mask": beats_mask,
        "beats_valid_ratio": torch.tensor([b["beats_valid_ratio"] for b in batch], dtype=torch.float32),
        "duration": torch.tensor([b["duration"] for b in batch], dtype=torch.float32),
        "coarse_target": torch.stack([b["coarse_target"] for b in batch]),
        "fine_clip_target": torch.stack([b["fine_clip_target"] for b in batch]),
        "event_clip_target": torch.stack([b["event_clip_target"] for b in batch]),
        "event_spans": [b["event_spans"] for b in batch],
        "tier": torch.tensor([b["tier"] for b in batch], dtype=torch.long),
        "src_idx": torch.tensor([b["src_idx"] for b in batch], dtype=torch.long),
        "full_idx": torch.tensor([b["full_idx"] for b in batch], dtype=torch.long),
    }

In [8]:
# =============================================================================
# CELL: Model -- accepts beats_valid_ratio and gates fusion on valid region
# =============================================================================
class VaaniLiteSED(nn.Module):
    def __init__(self, beats_model=None, n_fine=1, n_coarse=7,
                 beats_dim=768, proj_dim=192, gru_hidden=128, gru_layers=2,
                 mel_frontend=None):
        super().__init__()
        self.beats = beats_model; self.mel_frontend = mel_frontend
        self.beats_dim = beats_dim; self.proj_dim = proj_dim
        self.dfd_lka = DFDLKAEncoder(channels=(32, 64, 128, 192))
        self.beats_proj = nn.Linear(beats_dim, proj_dim)
        self.fdy_proj = nn.Linear(192, proj_dim)
        self.fuse_gate = nn.Sequential(nn.Linear(proj_dim * 2, proj_dim), nn.Sigmoid())
        self.fuse_norm = nn.LayerNorm(proj_dim)
        self.gru = nn.GRU(proj_dim, gru_hidden, num_layers=gru_layers,
                          batch_first=True, bidirectional=True, dropout=0.10)
        gru_out = 2 * gru_hidden
        self.gru_norm = nn.LayerNorm(gru_out)
        self.temporal_refine = TemporalRefinement(gru_out)
        self.event_frame_head = nn.Linear(gru_out, 1)
        self.event_onset_head = nn.Linear(gru_out, 1)
        self.event_offset_head = nn.Linear(gru_out, 1)
        self.event_attn = nn.Conv1d(gru_out, 1, kernel_size=1)
        self.event_clip_proj = nn.Linear(gru_out, 1)
        self.fine_attn = nn.Conv1d(gru_out, n_fine, kernel_size=1)
        self.fine_clip_proj = nn.Linear(gru_out, n_fine)
        self.coarse_attn = nn.Conv1d(gru_out, n_coarse, kernel_size=1)
        self.coarse_proj = nn.Linear(gru_out, n_coarse)

    def _beats_features(self, waveform, beats_cached):
        if beats_cached is not None: return beats_cached
        if self.beats is None: raise RuntimeError("No cached BEATs and no live BEATs model.")
        with torch.no_grad():
            feat, _, _ = beats_extract_temporal(self.beats, waveform)
        return feat

    def forward(self, waveform, beats_cached=None, beats_mask=None,
                beats_valid_ratio=None, mel=None):
        if waveform.ndim == 3: waveform = waveform.squeeze(1)
        beats_feat = self._beats_features(waveform, beats_cached)
        beats_feat = self.beats_proj(beats_feat)
        if mel is None:
            if self.mel_frontend is None: raise RuntimeError("mel_frontend required when mel=None")
            mel = self.mel_frontend(waveform)
        fdy = self.fdy_proj(self.dfd_lka(mel.unsqueeze(1)))
        T = fdy.shape[1]
        if beats_feat.shape[1] != T:
            beats_feat = F.interpolate(beats_feat.transpose(1, 2), size=T,
                                       mode="linear", align_corners=False).transpose(1, 2)
        gate = self.fuse_gate(torch.cat([beats_feat, fdy], dim=-1))
        # Gate fusion on the BEATs-valid region (short clips).
        if beats_valid_ratio is not None:
            arange = torch.arange(T, device=waveform.device, dtype=torch.float32).unsqueeze(0)
            valid_T = beats_valid_ratio.unsqueeze(1) * T
            frame_valid = (arange < valid_T).float()
            gate = gate * frame_valid.unsqueeze(-1)
        x = self.fuse_norm(gate * beats_feat + (1.0 - gate) * fdy)
        x, _ = self.gru(x); x = self.gru_norm(x); x = self.temporal_refine(x)
        event_frame_logits = self.event_frame_head(x)
        event_onset_logits = self.event_onset_head(x)
        event_offset_logits = self.event_offset_head(x)
        xf = x.transpose(1, 2)
        event_attn = F.softmax(self.event_attn(xf), dim=-1)
        event_pool = torch.einsum("bct,btd->bcd", event_attn, x)
        event_clip_logits = (event_pool * self.event_clip_proj.weight).sum(-1) + self.event_clip_proj.bias
        fine_attn = F.softmax(self.fine_attn(xf), dim=-1)
        coarse_attn = F.softmax(self.coarse_attn(xf), dim=-1)
        fine_pool = torch.einsum("bct,btd->bcd", fine_attn, x)
        coarse_pool = torch.einsum("bct,btd->bcd", coarse_attn, x)
        fine_clip_logits = torch.einsum("bcd,cd->bc", fine_pool, self.fine_clip_proj.weight) + self.fine_clip_proj.bias
        coarse_logits = torch.einsum("bcd,cd->bc", coarse_pool, self.coarse_proj.weight) + self.coarse_proj.bias
        return {"event_frame_logits": event_frame_logits,
                "event_onset_logits": event_onset_logits,
                "event_offset_logits": event_offset_logits,
                "event_clip_logits": event_clip_logits,
                "fine_clip_logits": fine_clip_logits,
                "coarse_logits": coarse_logits,
                "encoder_time_steps": T}

In [9]:
# =============================================================================
# CELL: Losses (unchanged)
# =============================================================================
class AsymmetricFocalLoss(nn.Module):
    def __init__(self, gamma_neg=4.0, gamma_pos=1.0, clip=0.05):
        super().__init__()
        self.gamma_neg = gamma_neg; self.gamma_pos = gamma_pos; self.clip = clip
    def forward(self, logits, targets, mask=None):
        probs = torch.sigmoid(logits)
        if self.clip > 0:
            xs_neg = (1.0 - probs + self.clip).clamp(max=1.0)
            probs = probs * targets + xs_neg * (1.0 - targets)
        pt = probs * targets + (1.0 - probs) * (1.0 - targets)
        gamma = self.gamma_pos * targets + self.gamma_neg * (1.0 - targets)
        loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none") * (1.0 - pt).pow(gamma)
        if mask is not None:
            m = mask.unsqueeze(-1).expand_as(loss) if mask.ndim == loss.ndim - 1 else mask
            return (loss * m).sum() / m.sum().clamp_min(1.0)
        return loss.mean()

def masked_bce(logits, targets, mask=None):
    loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    if mask is None: return loss.mean()
    m = mask.unsqueeze(-1).expand_as(loss) if mask.ndim == loss.ndim - 1 else mask
    return (loss * m).sum() / m.sum().clamp_min(1.0)

def classwise_attention_bce(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets)

def build_event_targets(event_spans_batch, batch_size, T, durations, device,
                        window_seconds=8.0, safe_margin_seconds=None):
    frame_target = torch.zeros(batch_size, T, 1, device=device)
    onset_target = torch.zeros(batch_size, T, 1, device=device)
    offset_target = torch.zeros(batch_size, T, 1, device=device)
    positive_mask = torch.zeros(batch_size, T, device=device)
    valid_mask = torch.zeros(batch_size, T, device=device)
    safe_negative_mask = torch.zeros(batch_size, T, device=device)
    centers = (torch.arange(T, device=device, dtype=torch.float32) + 0.5) * (window_seconds / T)
    for b in range(batch_size):
        dur = max(float(durations[b]), 1e-6)
        valid_mask[b] = (centers < float(durations[b])).float()
        spans = event_spans_batch[b]
        for start, end in spans:
            start = float(start); end = float(end)
            active = (centers >= start) & (centers < end)
            frame_target[b, active, 0] = 1.0; positive_mask[b, active] = 1.0
            sigma = max(dur / T, 0.03)
            on = torch.exp(-0.5 * ((centers - start) / sigma).pow(2))
            off = torch.exp(-0.5 * ((centers - end) / sigma).pow(2))
            onset_target[b, :, 0] = torch.maximum(onset_target[b, :, 0], on)
            offset_target[b, :, 0] = torch.maximum(offset_target[b, :, 0], off)
        if safe_margin_seconds is not None:
            if not spans:
                safe_negative_mask[b] = valid_mask[b]
            else:
                md = torch.full((T,), float("inf"), device=device)
                for start, end in spans:
                    start = float(start); end = float(end)
                    inside = (centers >= start) & (centers < end)
                    d = torch.minimum((centers - start).abs(), (centers - end).abs())
                    d = torch.where(inside, torch.zeros_like(d), d)
                    md = torch.minimum(md, d)
                safe_negative_mask[b] = valid_mask[b] * (md > safe_margin_seconds).float()
    return frame_target, onset_target, offset_target, valid_mask, positive_mask, safe_negative_mask

def consistency_loss(student_logits, teacher_logits, valid_mask, confidence_floor=0.60):
    s = torch.sigmoid(student_logits); t = torch.sigmoid(teacher_logits).detach()
    conf = torch.maximum(t, 1.0 - t)
    conf = ((conf - confidence_floor) / max(1.0 - confidence_floor, 1e-6)).clamp(0, 1)
    w = valid_mask.unsqueeze(-1) * conf
    return ((s - t).pow(2) * w).sum() / w.sum().clamp_min(1.0)

def hierarchical_consistency(fine_clip_logits, coarse_logits, fine_to_coarse_idx):
    fine_p = torch.sigmoid(fine_clip_logits); coarse_p = torch.sigmoid(coarse_logits)
    penalties = []
    for fi, ci in enumerate(fine_to_coarse_idx.tolist()):
        penalties.append(F.relu(fine_p[:, fi] - coarse_p[:, ci]).mean())
    return torch.stack(penalties).mean() if penalties else fine_clip_logits.new_tensor(0.0)

In [10]:
# =============================================================================
# CELL: Mean teacher + decoder + scorer
# =============================================================================
@torch.no_grad()
def update_teacher(student, teacher, decay=0.999):
    for sp, tp in zip(student.parameters(), teacher.parameters()):
        if tp is sp: continue
        tp.data.mul_(decay).add_(sp.data, alpha=1.0 - decay)
    for sb, tb in zip(student.buffers(), teacher.buffers()):
        if tb is sb: continue
        tb.copy_(sb)

def freeze_model(model):
    for p in model.parameters(): p.requires_grad_(False)
    model.eval()

def smooth_probs(probs_1d, window):
    if window is None or window <= 1: return probs_1d
    k = int(window)
    if k % 2 == 0: k += 1
    if k >= len(probs_1d): k = len(probs_1d) - (1 - len(probs_1d) % 2)
    if k < 3: return probs_1d
    return medfilt(probs_1d, kernel_size=k)

def snap_to_peak(probs, idx, radius):
    if probs is None or radius <= 0: return idx
    lo = max(0, idx - radius); hi = min(len(probs), idx + radius + 1)
    if hi <= lo: return idx
    return lo + int(np.argmax(probs[lo:hi]))

def decode_events_single(frame_probs, fps, threshold=0.5, smooth_window=5,
                         min_duration=0.15, merge_gap=0.10,
                         onset_probs=None, offset_probs=None,
                         snap_radius_seconds=0.15):
    probs = smooth_probs(np.asarray(frame_probs, dtype=np.float32), smooth_window)
    binary = (probs >= threshold).astype(np.int32)
    raw = []; in_ev, s_idx = False, 0
    for i, v in enumerate(binary):
        if v and not in_ev: in_ev, s_idx = True, i
        elif not v and in_ev: in_ev = False; raw.append((s_idx, i))
    if in_ev: raw.append((s_idx, len(binary)))
    radius = max(1, int(round(snap_radius_seconds * fps)))
    snapped = []
    for s, e in raw:
        ss = snap_to_peak(onset_probs, s, radius)
        es = snap_to_peak(offset_probs, e, radius)
        if es <= ss: ss, es = s, e
        snapped.append((ss, es))
    merged = []
    for s, e in snapped:
        if merged and (s - merged[-1][1]) / fps <= merge_gap:
            merged[-1] = (merged[-1][0], max(merged[-1][1], e))
        else: merged.append((s, e))
    out = []
    for s, e in merged:
        onset, offset = s / fps, e / fps
        if offset - onset >= min_duration: out.append((float(onset), float(offset)))
    return out

def match_events(ref_events, pred_events, tolerance_frac=0.20, min_tol=0.05):
    mr, mp = set(), set(); cand = []
    for ri, (ro, rf) in enumerate(ref_events):
        tol = max(tolerance_frac * (rf - ro), min_tol)
        for pi, (po, pf) in enumerate(pred_events):
            if abs(po - ro) <= tol and abs(pf - rf) <= tol:
                cand.append((abs(po - ro) + abs(pf - rf), ri, pi))
    for _, ri, pi in sorted(cand):
        if ri not in mr and pi not in mp: mr.add(ri); mp.add(pi)
    tp = len(mr)
    return tp, len(pred_events) - tp, len(ref_events) - tp

def event_based_f1(ref_data, pred_data):
    TP = FP = FN = 0
    for cid, re in ref_data.items():
        tp, fp, fn = match_events(re, pred_data.get(cid, []))
        TP += tp; FP += fp; FN += fn
    for cid in pred_data:
        if cid not in ref_data: FP += len(pred_data[cid])
    p = TP / (TP + FP) if (TP + FP) else 0.0
    r = TP / (TP + FN) if (TP + FN) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return f1, p, r

def events_to_frames(events, max_time, frame_len=0.01):
    n = int(max_time / frame_len) + 1
    m = np.zeros(n, dtype=np.int32)
    for on, off in events:
        i0 = int(on / frame_len); i1 = min(int(off / frame_len) + 1, n)
        if i1 > i0: m[i0:i1] = 1
    return m

def segment_dice(ref_data, pred_data):
    scores = []
    for cid, re in ref_data.items():
        pe = pred_data.get(cid, []); all_ev = re + pe
        if not all_ev: scores.append(1.0); continue
        mt = max(off for _, off in all_ev) + 0.5
        r = events_to_frames(re, mt); p = events_to_frames(pe, mt)
        i = int((r & p).sum()); t = int(r.sum() + p.sum())
        scores.append(1.0 if t == 0 else 2.0 * i / t)
    return sum(scores) / len(scores) if scores else 0.0

def official_combined_score(ref_data, pred_data):
    f1, p, r = event_based_f1(ref_data, pred_data)
    dice = segment_dice(ref_data, pred_data)
    return {"event_f1": f1, "precision": p, "recall": r,
            "segment_dice": dice, "combined": f1 + dice}

In [11]:
# =============================================================================
# CELL: Split / label space / priors (index-aware)
# =============================================================================
def split_train_val(dataset, val_fraction=0.10, seed=42, group_col="district"):
    groups = dataset[group_col]
    gss = GroupShuffleSplit(n_splits=1, test_size=val_fraction, random_state=seed)
    ia = np.arange(len(dataset))
    tr, va = next(gss.split(ia, groups=groups))
    return tr.tolist(), va.tolist()

def build_label_space(ds, indices=None, min_tag_count_to_keep=5):
    idxs = list(range(len(ds))) if indices is None else list(indices)
    tag_counts = Counter(); tag_to_coarse_votes = defaultdict(Counter)
    try:
        subcat_col = ds["NoiseSubCategoryTimeStamp"]; cats_col = ds["NoiseCategory"]
        use_cols = True
    except Exception:
        use_cols = False
    for i in idxs:
        if use_cols:
            events = subcat_col[i] or []; cats = cats_col[i] or []
        else:
            s = ds[i]; events = s.get("NoiseSubCategoryTimeStamp") or []; cats = s.get("NoiseCategory") or []
        for e in events:
            raw = e.get("tag", "")
            if not raw: continue
            tag_counts[raw] += 1
            for c in cats:
                if c in COARSE_TO_IDX: tag_to_coarse_votes[raw][c] += 1
    raw_to_canonical, cluster_report = build_tag_clusters(tag_counts)
    canon_counts = Counter(); canon_coarse_votes = defaultdict(Counter)
    for raw, cnt in tag_counts.items():
        canon = raw_to_canonical[raw]; canon_counts[canon] += cnt
        for c, v in tag_to_coarse_votes[raw].items(): canon_coarse_votes[canon][c] += v
    canonical_tags = sorted(t for t, c in canon_counts.items() if c >= min_tag_count_to_keep)
    f2c = torch.zeros(len(canonical_tags), dtype=torch.long)
    for i, tag in enumerate(canonical_tags):
        v = canon_coarse_votes.get(tag)
        f2c[i] = COARSE_TO_IDX[v.most_common(1)[0][0]] if v else 0
    return canonical_tags, raw_to_canonical, f2c, cluster_report

def estimate_global_duration_prior(ds, indices=None):
    idxs = list(range(len(ds))) if indices is None else list(indices)
    try:
        qcol = ds["annotationQuality"]; scol = ds["NoiseSubCategoryTimeStamp"]; use_cols = True
    except Exception:
        use_cols = False
    out = []
    for i in idxs:
        if use_cols:
            if qcol[i] != "verified_timestamps": continue
            events = scol[i] or []
        else:
            s = ds[i]
            if s.get("annotationQuality") != "verified_timestamps": continue
            events = s.get("NoiseSubCategoryTimeStamp") or []
        for e in events:
            try: d = float(e["end"]) - float(e["start"])
            except (TypeError, ValueError, KeyError): continue
            if d > 0: out.append(d)
    return float(np.median(out)) if out else 1.0

def compute_adaptive_window(median_duration, fps, min_window=3, max_window=25):
    w = int(round(median_duration * fps * 0.5))
    w = max(min_window, min(max_window, w))
    return w + 1 if w % 2 == 0 else w

def weak_augment(mel): return apply_mel_specaugment(mel, 1, 1, 0.05, 8)
def strong_augment(mel): return apply_mel_specaugment(mel, 2, 2, 0.12, 16)

TIER_LOSS_SCALE = {TIER_VERIFIED: 1.0, TIER_UNVERIFIED: 0.6,
                   TIER_WEAK_ONLY: 0.0, TIER_PSEUDO_BRONZE: 0.4}

def make_tier_scales(tier_tensor, device):
    scale = torch.zeros(tier_tensor.shape[0], device=device)
    for t, s in TIER_LOSS_SCALE.items(): scale[tier_tensor == t] = s
    return scale

In [12]:
# =============================================================================
# CELL: run_epoch -- strict cache, per-update warmup, beats_valid_ratio plumbed
# + mel computed in fp32 (same fix as BEATs extraction: FFT/log10 under fp16
#   autocast can produce inf/nan on quiet or degenerate clips)
# + hard stop with full diagnostics at the first non-finite loss
# =============================================================================
LOSS_WEIGHTS = dict(event_frame=3.0, onset=0.5, offset=0.5, event_clip=0.5,
                    fine_clip=0.3, coarse=0.3, hierarchy=0.05, consistency=1.0)

DEBUG_NAN_SNAPSHOT_PATH = "/kaggle/working/debug_nan_snapshot.pt"

def run_epoch(model, teacher, loader, optimizer, scaler, device,
              fine_to_coarse_idx, window_seconds, use_mean_teacher=False,
              teacher_decay=0.999, train=True, loss_weights=None,
              safe_margin_seconds=None, silver_negative_weight=0.5,
              grad_accum_steps=1, warmup_scheduler=None, warmup_state=None,
              log_warmup=True):
    lw = loss_weights or LOSS_WEIGHTS
    model.train(mode=train)
    if teacher is not None: teacher.eval()
    totals = defaultdict(float); n_batches = 0
    optimizer.zero_grad(set_to_none=True)
    if warmup_state is None:
        warmup_state = {"updates": 0, "total": 0}

    def _do_optimizer_step():
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        prev_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        step_happened = scaler.get_scale() >= prev_scale
        if step_happened:
            warmup_state["updates"] = warmup_state.get("updates", 0) + 1
            n = warmup_state["updates"]; total = warmup_state.get("total", 0)
            if warmup_scheduler is not None and total > 0 and n <= total:
                warmup_scheduler.step()
            if log_warmup and n <= 5:
                lr_now = optimizer.param_groups[0]["lr"]
                print(f"  [warmup] opt-update {n}/{total} lr={lr_now:.6e}")
        return step_happened

    def _dump_nan_and_stop(reason, batch, extra=None):
        print(f"\n[NaN-STOP] {reason}")
        print(f"  step_i={step_i}  batch full_idx={batch['full_idx'].tolist()}")
        print(f"  batch tier={batch['tier'].tolist()}")
        print(f"  batch duration={batch['duration'].tolist()}")
        if extra:
            print(f"  per-component finiteness: {extra}")
        torch.save({
            "model": model.state_dict(),
            "batch": {k: (v if not torch.is_tensor(v) else v.cpu())
                      for k, v in batch.items() if k != "event_spans"},
            "event_spans": batch["event_spans"],
            "step_i": step_i,
            "reason": reason,
        }, DEBUG_NAN_SNAPSHOT_PATH)
        print(f"  snapshot saved to {DEBUG_NAN_SNAPSHOT_PATH}")
        raise RuntimeError(f"Stopping: {reason} (see snapshot for exact repro batch)")

    for step_i, batch in enumerate(loader):
        waveform = batch["waveform"].to(device, non_blocking=True)
        beats_padded = batch["beats_padded"].to(device, non_blocking=True)
        beats_valid_ratio = batch["beats_valid_ratio"].to(device)
        duration = batch["duration"].to(device)
        coarse_target = batch["coarse_target"].to(device)
        fine_clip_target = batch["fine_clip_target"].to(device)
        event_clip_target = batch["event_clip_target"].to(device).unsqueeze(-1)
        event_spans = batch["event_spans"]
        tier = batch["tier"].to(device)
        B = waveform.shape[0]
        tier_scale = make_tier_scales(tier, device)

        beats_cached = beats_padded

        # ---- mel in fp32, same reasoning as the BEATs fp32 fix -------------
        with autocast(device_type="cuda", enabled=False):
            mel_clean = model.mel_frontend(waveform.float())
        if not torch.isfinite(mel_clean).all():
            bad = (~torch.isfinite(mel_clean).flatten(1).all(dim=1)).nonzero(as_tuple=True)[0]
            _dump_nan_and_stop(
                f"mel_frontend produced non-finite output for local batch rows {bad.tolist()}",
                batch)

        with autocast(device_type="cuda", enabled=(device.type == "cuda")):
            mel_student = weak_augment(mel_clean) if train else mel_clean
            out = model(waveform, beats_cached=beats_cached,
                        beats_valid_ratio=beats_valid_ratio, mel=mel_student)
            T = out["encoder_time_steps"]
            frame_t, onset_t, offset_t, valid_mask, pos_mask, safe_neg = build_event_targets(
                event_spans, B, T, duration, device, window_seconds=window_seconds,
                safe_margin_seconds=safe_margin_seconds)
            is_verified = (tier == TIER_VERIFIED).unsqueeze(-1)
            non_gold = pos_mask + silver_negative_weight * safe_neg * (1.0 - pos_mask)
            strong_base = torch.where(is_verified, valid_mask, non_gold)
            strong_mask = strong_base * tier_scale.unsqueeze(-1)
            boundary_mask = valid_mask * tier_scale.unsqueeze(-1)
            loss_event_frame = AsymmetricFocalLoss()(out["event_frame_logits"], frame_t, mask=strong_mask)
            loss_onset = masked_bce(out["event_onset_logits"], onset_t, mask=boundary_mask)
            loss_offset = masked_bce(out["event_offset_logits"], offset_t, mask=boundary_mask)
            loss_event_clip = F.binary_cross_entropy_with_logits(out["event_clip_logits"], event_clip_target)
            loss_fine_clip = classwise_attention_bce(out["fine_clip_logits"], fine_clip_target)
            loss_coarse = classwise_attention_bce(out["coarse_logits"], coarse_target)
            loss_hier = hierarchical_consistency(out["fine_clip_logits"], out["coarse_logits"], fine_to_coarse_idx)

            # ---- per-component finiteness check, BEFORE summing ------------
            components = {
                "event_frame": loss_event_frame, "onset": loss_onset,
                "offset": loss_offset, "event_clip": loss_event_clip,
                "fine_clip": loss_fine_clip, "coarse": loss_coarse, "hier": loss_hier,
            }
            bad_components = {k: bool(torch.isfinite(v)) for k, v in components.items()}
            if not all(bad_components.values()):
                _dump_nan_and_stop(
                    "one or more loss components is non-finite before summing",
                    batch, extra=bad_components)

            loss = (lw["event_frame"] * loss_event_frame
                    + lw["onset"] * loss_onset + lw["offset"] * loss_offset
                    + lw["event_clip"] * loss_event_clip
                    + lw["fine_clip"] * loss_fine_clip
                    + lw["coarse"] * loss_coarse
                    + lw["hierarchy"] * loss_hier)
            loss_consist = frame_t.new_tensor(0.0)
            if use_mean_teacher and teacher is not None:
                with torch.no_grad():
                    with autocast(device_type="cuda", enabled=False):
                        mel_teacher = strong_augment(mel_clean)
                    t_out = teacher(waveform, beats_cached=beats_cached,
                                    beats_valid_ratio=beats_valid_ratio,
                                    mel=mel_teacher)
                loss_consist = consistency_loss(out["event_frame_logits"],
                                                t_out["event_frame_logits"], valid_mask)
                if not torch.isfinite(loss_consist):
                    _dump_nan_and_stop("consistency loss is non-finite", batch)
                loss = loss + lw["consistency"] * loss_consist
            loss = loss / grad_accum_steps

        if not torch.isfinite(loss):
            _dump_nan_and_stop("combined loss is non-finite after weighting", batch,
                               extra=bad_components)

        if train:
            scaler.scale(loss).backward()
            if (step_i + 1) % grad_accum_steps == 0:
                _do_optimizer_step()
                if use_mean_teacher and teacher is not None:
                    update_teacher(model, teacher, decay=teacher_decay)

        totals["loss"] += float(loss.item()) * grad_accum_steps
        totals["event_frame"] += float(loss_event_frame.item())
        totals["onset"] += float(loss_onset.item())
        totals["offset"] += float(loss_offset.item())
        totals["event_clip"] += float(loss_event_clip.item())
        totals["fine_clip"] += float(loss_fine_clip.item())
        totals["coarse"] += float(loss_coarse.item())
        totals["hier"] += float(loss_hier.item())
        totals["consistency"] += float(loss_consist.item())
        n_batches += 1

    if train and (len(loader) % grad_accum_steps) != 0:
        _do_optimizer_step()
        if use_mean_teacher and teacher is not None:
            update_teacher(model, teacher, decay=teacher_decay)

    return {k: v / max(n_batches, 1) for k, v in totals.items()}

In [13]:
# =============================================================================
# CELL: Inference utilities -- strict cache lookup
# =============================================================================
@torch.no_grad()
def get_model_time_grid(model, device, window_seconds, sample_rate):
    model.eval()
    dummy = torch.zeros(1, int(round(window_seconds * sample_rate)), device=device)
    out = model(dummy)
    T = out["encoder_time_steps"]
    return T, T / window_seconds

def _ref_events_for_sample(sample):
    evs = []
    for e in (sample.get("NoiseSubCategoryTimeStamp") or []):
        try: s = float(e["start"]); en = float(e["end"])
        except (TypeError, ValueError, KeyError): continue
        if en > s: evs.append((s, en))
    return evs

def _window_starts(total_samples, win, stride):
    if total_samples <= win: return [0]
    starts = list(range(0, total_samples - win + 1, stride))
    if starts[-1] + win < total_samples: starts.append(total_samples - win)
    return starts

@torch.no_grad()
def _windowed_logit_tracks(model, wav, sample_rate, device, T_win, fps,
                            window_seconds, stride_seconds,
                            beats_full_np=None, full_duration=None):
    total = wav.numel(); duration_sec = total / sample_rate
    win = int(round(window_seconds * sample_rate)); stride = int(round(stride_seconds * sample_rate))
    total_frames = int(round(duration_sec * fps)) + T_win + 1
    sums = {k: np.zeros(total_frames, dtype=np.float64) for k in ("frame", "onset", "offset")}
    wsum = np.zeros(total_frames, dtype=np.float64)
    for s0 in _window_starts(total, win, stride):
        chunk = wav[s0:s0+win]
        pad_len = win - chunk.numel()
        if pad_len > 0: chunk = F.pad(chunk, (0, pad_len))
        chunk = chunk.unsqueeze(0).to(device)
        beats_cached = None
        if beats_full_np is not None and full_duration is not None:
            T_full = beats_full_np.shape[0]
            win_start_s = s0 / sample_rate
            win_end_s   = (s0 + win) / sample_rate
            # Use the same patch-time mapping as the dataset.
            p0 = int(np.floor((win_start_s - BEATS_PATCH_OFFSET_SEC) / BEATS_PATCH_HOP_SEC))
            p1 = int(np.ceil((win_end_s   - BEATS_PATCH_OFFSET_SEC) / BEATS_PATCH_HOP_SEC)) + 1
            p0 = max(0, min(p0, T_full - 1)); p1 = max(p0 + 1, min(p1, T_full))
            beats_cached = torch.from_numpy(beats_full_np[p0:p1]).float().unsqueeze(0).to(device)
        out = model(chunk, beats_cached=beats_cached)
        frame_l = out["event_frame_logits"][0, :, 0].detach().cpu().numpy()
        onset_l = out["event_onset_logits"][0, :, 0].detach().cpu().numpy()
        offset_l = out["event_offset_logits"][0, :, 0].detach().cpu().numpy()
        off = int(round((s0 / sample_rate) * fps))
        end = min(off + T_win, total_frames); L = end - off
        if L > 0:
            sums["frame"][off:end] += frame_l[:L]
            sums["onset"][off:end] += onset_l[:L]
            sums["offset"][off:end] += offset_l[:L]
            wsum[off:end] += 1.0
    valid_len = min(int(round(duration_sec * fps)) + 1, total_frames)
    w = np.maximum(wsum[:valid_len], 1e-6)
    merged = {k: sums[k][:valid_len] / w for k in sums}
    return {k: 1.0 / (1.0 + np.exp(-merged[k])) for k in merged}

@torch.no_grad()
def compute_val_probabilities(model, parent_hf, full_indices, device, sample_rate,
                              beats_cache_dir=None, wav_cache_dir=None, label_table=None,
                              window_seconds=8.0, stride_seconds=4.0, max_clips=None):
    model.eval()
    T_win, fps = get_model_time_grid(model, device, window_seconds, sample_rate)
    n = len(full_indices) if max_clips is None else min(max_clips, len(full_indices))
    ref_data, probs_by_clip = {}, {}
    for i in range(n):
        full_idx = int(full_indices[i])
        # CHANGED: cache-aware waveform load (see _load_waveform) instead of
        # a live decode+resample every epoch this runs.
        wav = _load_waveform(full_idx, parent_hf, sample_rate, wav_cache_dir)
        full_dur = wav.numel() / sample_rate
        beats_full_np = None
        if beats_cache_dir is not None:
            p = resolve_cache_path(full_idx, beats_cache_dir)
            if p is None:
                raise RuntimeError(f"Missing cache for full_idx={full_idx} during val inference "
                                   f"(checked {_normalize_cache_dirs(beats_cache_dir)})")
            beats_full_np = np.load(p).astype(np.float32)
        merged = _windowed_logit_tracks(model, wav, sample_rate, device, T_win, fps,
                                        window_seconds, stride_seconds,
                                        beats_full_np=beats_full_np, full_duration=full_dur)
        # CHANGED: label_table avoids the row access that would otherwise
        # decode audio just to read the clip id / ref events.
        if label_table is not None:
            cid = str(label_table["id"][full_idx] or full_idx)
            ref_data[cid] = _ref_events_for_full_idx(full_idx, label_table)
        else:
            sample = parent_hf[full_idx]
            cid = str(sample.get("id") or sample.get("clip_id") or full_idx)
            ref_data[cid] = _ref_events_for_sample(sample)
        probs_by_clip[cid] = merged
    return ref_data, probs_by_clip, fps

def _decode_clip(cached, fps, thr, sw, md, mg, sr):
    return decode_events_single(cached["frame"], fps, threshold=thr, smooth_window=sw,
                                min_duration=md, merge_gap=mg,
                                onset_probs=cached["onset"], offset_probs=cached["offset"],
                                snap_radius_seconds=sr)

def grid_search_decode_params(ref_data, probs_by_clip, fps, median_duration,
                              thresholds=(0.3, 0.4, 0.5, 0.6, 0.7),
                              smooth_windows=None,
                              min_duration_fracs=(0.1, 0.2, 0.3),
                              merge_gaps=(0.05, 0.10, 0.15, 0.20),
                              snap_radius_seconds_options=(0.10, 0.25)):
    if smooth_windows is None:
        base_w = compute_adaptive_window(median_duration, fps)
        smooth_windows = sorted(set([max(3, base_w // 2), base_w, base_w * 2]))
    best = {"combined": -1.0}
    for thr in thresholds:
        for sw in smooth_windows:
            for mdf in min_duration_fracs:
                md = max(0.05, mdf * median_duration)
                for g in merge_gaps:
                    for sr in snap_radius_seconds_options:
                        pred = {cid: _decode_clip(c, fps, thr, sw, md, g, sr)
                                for cid, c in probs_by_clip.items()}
                        score = official_combined_score(ref_data, pred)
                        if score["combined"] > best["combined"]:
                            best = {**score, "threshold": thr, "smooth_window": sw,
                                    "min_duration": md, "merge_gap": g,
                                    "snap_radius_seconds": sr}
    return best

@torch.no_grad()
def generate_pseudo_labels(teacher_model, parent_hf, bronze_full_indices, sample_rate, device,
                           decode_params, beats_cache_dir=None, wav_cache_dir=None,
                           window_seconds=8.0, stride_seconds=4.0,
                           min_confidence=0.65, max_clips=None):
    teacher_model.eval()
    T_win, fps = get_model_time_grid(teacher_model, device, window_seconds, sample_rate)
    indices = [int(i) for i in bronze_full_indices]
    if max_clips is not None: indices = indices[:max_clips]
    overrides = {}
    for fi in indices:
        # CHANGED: cache-aware waveform load instead of a live decode --
        # this only runs once per Bronze clip per stage, so lower total
        # impact than the training loader, but still free to fix the same way.
        wav = _load_waveform(fi, parent_hf, sample_rate, wav_cache_dir)
        full_dur = wav.numel() / sample_rate
        p = resolve_cache_path(fi, beats_cache_dir)
        if p is None:
            raise RuntimeError(f"Missing cache for full_idx={fi} during pseudo-labeling "
                               f"(checked {_normalize_cache_dirs(beats_cache_dir)})")
        beats_full_np = np.load(p).astype(np.float32)
        merged = _windowed_logit_tracks(teacher_model, wav, sample_rate, device, T_win, fps,
                                        window_seconds, stride_seconds,
                                        beats_full_np=beats_full_np, full_duration=full_dur)
        spans = decode_events_single(
            merged["frame"], fps, threshold=decode_params["threshold"],
            smooth_window=decode_params["smooth_window"],
            min_duration=decode_params["min_duration"], merge_gap=decode_params["merge_gap"],
            onset_probs=merged["onset"], offset_probs=merged["offset"],
            snap_radius_seconds=decode_params.get("snap_radius_seconds", 0.15))
        accepted = []
        for s, e in spans:
            si = int(round(s * fps)); ei = min(int(round(e * fps)), len(merged["frame"]) - 1)
            peak = float(merged["frame"][si:ei+1].max()) if ei >= si else 0.0
            if peak >= min_confidence: accepted.append({"start": s, "end": e})
        if accepted: overrides[fi] = {"events": accepted}
    return overrides

In [14]:
# =============================================================================
# CELL: Config
# =============================================================================
CONFIG = dict(
    seed=42, val_fraction=0.10, sample_rate=16000, crop_seconds=8.0, n_mels=128,
    min_tag_count_to_keep=5, batch_size=8, grad_accum_steps=2,
    num_workers=4,  # CHANGED: was 2 -- decode is now cached, workers just do np.load + cropping
    stage1_epochs=15, stage2_epochs=10, stage3_epochs=8,
    lr_stage1=3e-4, lr_stage2=8e-5, lr_stage3=5e-5,
    lr_warmup_steps=500, teacher_decay=0.999, silver_negative_weight=0.5,
    max_eval_clips=300,
)

In [15]:
# =============================================================================
# CELL: train_one_stage -- exhaustive cache validation; per-update warmup
# =============================================================================
def train_one_stage(model, teacher, parent_hf, tier_indices_global,
                    val_full_indices, canonical_tags, raw_to_canonical,
                    fine_to_coarse_idx, pseudo_overrides_global, median_duration,
                    device, config, stage_name,
                    include_bronze, use_mean_teacher, epochs, lr,
                    checkpoint_path=None, wav_cache_dir=None, label_table=None):
    print(f"\n=== {stage_name} ===")
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                  lr=lr, weight_decay=1e-4)
    scaler = GradScaler(enabled=(device.type == "cuda"))

    subset_full = list(tier_indices_global["gold"]) + list(tier_indices_global["silver"])
    if include_bronze: subset_full = subset_full + list(tier_indices_global["bronze"])
    subset_full = [int(i) for i in subset_full]
    print(f"  training subset size: {len(subset_full)} full-dataset rows")

    # STRICT: every required index must have a cache entry.
    _validate_cache_exhaustive(subset_full, BEATS_CACHE_DIRS, label=stage_name)

    dataset = VaaniNoiseDataset(
        parent_hf=parent_hf, full_indices=subset_full,
        raw_to_canonical=raw_to_canonical, canonical_tags=canonical_tags,
        fine_to_coarse_idx=fine_to_coarse_idx,
        beats_cache_dir=BEATS_CACHE_DIRS,
        wav_cache_dir=wav_cache_dir, label_table=label_table,
        pseudo_overrides=pseudo_overrides_global,
        crop_seconds=config["crop_seconds"], sample_rate=config["sample_rate"],
        training=True)

    print("  [index-sanity] first 5 local->global mappings:")
    for li in range(min(5, len(dataset))):
        fi = int(dataset.full_indices[li])
        print(f"    local={li} full_idx={fi} cache={resolve_cache_path(fi, BEATS_CACHE_DIRS)}")
    dataset.verify_cache_alignment(n_samples=5)

    loader = DataLoader(dataset, batch_size=config["batch_size"], shuffle=True,
                        num_workers=config["num_workers"], collate_fn=collate_fn,
                        pin_memory=True, persistent_workers=config["num_workers"] > 0,
                        drop_last=True)

    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, end_factor=1.0,
        total_iters=max(1, int(config["lr_warmup_steps"])))
    warmup_state = {"updates": 0, "total": int(config["lr_warmup_steps"])}

    default_window = compute_adaptive_window(median_duration, 25.0)
    best_decode_params = dict(threshold=0.5, smooth_window=default_window,
                              min_duration=max(0.05, 0.2 * median_duration), merge_gap=0.10,
                              snap_radius_seconds=0.15)
    best_combined = -1.0
    ref_data = probs_by_clip = None; fps_probe = None
    safe_margin_seconds = 2.0 * median_duration

    for epoch in range(1, epochs + 1):
        train_stats = run_epoch(
            model, teacher, loader, optimizer, scaler, device,
            fine_to_coarse_idx=fine_to_coarse_idx, window_seconds=config["crop_seconds"],
            use_mean_teacher=use_mean_teacher, teacher_decay=config["teacher_decay"],
            train=True, safe_margin_seconds=safe_margin_seconds,
            silver_negative_weight=config["silver_negative_weight"],
            grad_accum_steps=config["grad_accum_steps"],
            warmup_scheduler=warmup_scheduler, warmup_state=warmup_state)

        print(f"[{stage_name}] epoch {epoch}/{epochs} "
              f"opt-updates={warmup_state['updates']}/{warmup_state['total']} "
              + " ".join(f"{k}={v:.4f}" for k, v in train_stats.items()))

        ref_data, probs_by_clip, fps_probe = compute_val_probabilities(
            model, parent_hf, val_full_indices, device, config["sample_rate"],
            beats_cache_dir=BEATS_CACHE_DIRS, wav_cache_dir=wav_cache_dir, label_table=label_table,
            window_seconds=config["crop_seconds"],
            stride_seconds=config["crop_seconds"] / 2,
            max_clips=config["max_eval_clips"])
        if epoch == 1:
            best_decode_params["smooth_window"] = compute_adaptive_window(median_duration, fps_probe)

        # NEW: direct confirmation of frame-head collapse (predicting no
        # event anywhere) instead of inferring it from a flat dev score --
        # see diagnosis: event_frame/event_clip loss -> 0 while fine_clip/
        # coarse keep improving normally means the localization head has
        # saturated, not converged.
        all_probs = np.concatenate([c["frame"] for c in probs_by_clip.values()])
        print(f"  [diag] frame-prob stats: mean={all_probs.mean():.4f} max={all_probs.max():.4f} "
              f"frac>0.5={(all_probs > 0.5).mean():.6f}")

        pred_data = {cid: _decode_clip(c, fps_probe,
                                       best_decode_params["threshold"],
                                       best_decode_params["smooth_window"],
                                       best_decode_params["min_duration"],
                                       best_decode_params["merge_gap"],
                                       best_decode_params["snap_radius_seconds"])
                     for cid, c in probs_by_clip.items()}
        n_empty_pred = sum(1 for v in pred_data.values() if len(v) == 0)
        print(f"  [diag] pred_data: {n_empty_pred}/{len(pred_data)} clips got zero predicted events")
        score = official_combined_score(ref_data, pred_data)
        print(f"  dev: Event-F1={score['event_f1']:.4f} Dice={score['segment_dice']:.4f} "
              f"Combined={score['combined']:.4f}")

        if score["combined"] > best_combined:
            best_combined = score["combined"]
            if checkpoint_path:
                torch.save({"model": model.state_dict(),
                            "teacher": teacher.state_dict() if teacher is not None else None,
                            "combined": best_combined, "canonical_tags": canonical_tags},
                           checkpoint_path)
                print(f"  saved {checkpoint_path} (combined={best_combined:.4f})")

    print(f"[{stage_name}] tuning decode params on {len(ref_data)} dev clips...")
    tuned = grid_search_decode_params(ref_data, probs_by_clip, fps_probe, median_duration)
    print(f"[{stage_name}] tuned: thr={tuned['threshold']} smooth={tuned['smooth_window']} "
          f"min_dur={tuned['min_duration']:.3f}s gap={tuned['merge_gap']:.3f}s "
          f"-> Combined={tuned['combined']:.4f}")

    if checkpoint_path and os.path.exists(checkpoint_path):
        ck = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"])
        if teacher is not None and ck.get("teacher") is not None:
            teacher.load_state_dict(ck["teacher"])
        print(f"  reloaded best checkpoint (combined={ck['combined']:.4f})")

    return {"best_combined": best_combined, "tuned_decode_params": tuned, "fps": fps_probe}

In [16]:
# =============================================================================
# CELL: main_three_stage -- full-cache validation up front
#
# Rows whose BEATs cache entry does not exist (build failed for corrupt/
# unreadable audio) are dropped from both splits up front. This is not a
# silent fallback -- it prints exactly how many were dropped and their
# indices, so any unexpected cache holes are still visible.
# =============================================================================
def main_three_stage(full_dataset, config, device):
    set_seed(config["seed"])
    train_idx, val_idx = split_train_val(full_dataset,
                                         val_fraction=config["val_fraction"],
                                         seed=config["seed"])

    # ---- Drop rows with no BEATs cache entry -------------------------------
    # build_beats_cache() skips rows that fail to load (corrupt audio,
    # unreadable bytes, missing path) and prints "[cache] failed idx=...".
    # Rather than rebuilding, we exclude those rows from the run. The
    # number dropped is printed explicitly so a large unexpected drop
    # (indicating real cache corruption) is immediately visible.
    import os as _os
    def _has_cache(i):
        return resolve_cache_path(i, BEATS_CACHE_DIRS) is not None

    n_train_before, n_val_before = len(train_idx), len(val_idx)
    dropped_train = [i for i in train_idx if not _has_cache(i)]
    dropped_val   = [i for i in val_idx   if not _has_cache(i)]
    if dropped_train or dropped_val:
        train_idx = [i for i in train_idx if _has_cache(i)]
        val_idx   = [i for i in val_idx   if _has_cache(i)]
        print(f"Dropped {len(dropped_train)} train + {len(dropped_val)} val rows "
              f"with no BEATs cache entry.")
        if dropped_train: print(f"  train dropped (first 10): {dropped_train[:10]}")
        if dropped_val:   print(f"  val   dropped (first 10): {dropped_val[:10]}")
        # Safety: refuse to continue if the drop is suspiciously large.
        # A healthy run should drop 0-5 rows. If hundreds are missing, the
        # cache is truncated/corrupt and training must not proceed silently.
        if len(dropped_train) > 100:
            raise RuntimeError(
                f"Refusing to proceed: {len(dropped_train)} train rows have no "
                f"cache entry. This indicates the cache is incomplete, not a "
                f"few bad clips. Rebuild the cache.")
    else:
        print("All train/val rows have BEATs cache entries.")

    print(f"train split: {len(train_idx)}, val split: {len(val_idx)} (full-dataset indices)")

    tier_indices_global = split_indices_by_tier(full_dataset, indices=train_idx)
    print("tier sizes (global):", {k: len(v) for k, v in tier_indices_global.items()})

    val_quality = full_dataset["annotationQuality"]
    val_full_gold = [int(i) for i in val_idx if val_quality[i] == "verified_timestamps"]
    print(f"val (Gold-only): {len(val_full_gold)} / {len(val_idx)}")

    # Union of every index the entire 3-stage curriculum will touch.
    # After the filter above, this must pass -- anything still missing
    # inside train_idx or val_full_gold is a real invariant violation.
    all_required = sorted(set(train_idx) | set(val_full_gold))
    _validate_cache_exhaustive(all_required, BEATS_CACHE_DIRS, label="main")

    canonical_tags, raw_to_canonical, f2c, cluster_report = build_label_space(
        full_dataset, indices=train_idx,
        min_tag_count_to_keep=config["min_tag_count_to_keep"])
    f2c = f2c.to(device)
    print(f"auxiliary fine classes: {len(canonical_tags)}")
    write_cluster_report_csv(cluster_report, "/kaggle/working/tag_clusters.csv")

    # NEW: columnar label pull, once. See build_label_table's docstring --
    # this is what lets the waveform cache actually pay off, since
    # per-row hf_dataset[i] access decodes audio regardless of which
    # fields you read off the row.
    label_table = build_label_table(full_dataset)
    print(f"label table built for {len(label_table['annotationQuality'])} rows")

    median_duration = estimate_global_duration_prior(full_dataset, indices=train_idx)
    print(f"median Gold duration (train): {median_duration:.3f}s")

    beats_model, beats_dim = load_beats_model(BEATS_CHECKPOINT, device)
    mel_frontend = MelFrontend(n_mels=config["n_mels"], sample_rate=config["sample_rate"],
                               max_seconds=config["crop_seconds"]).to(device)
    model = VaaniLiteSED(beats_model, n_fine=len(canonical_tags),
                         n_coarse=len(COARSE_CATEGORIES), beats_dim=beats_dim,
                         mel_frontend=mel_frontend).to(device)
    teacher = VaaniLiteSED(beats_model, n_fine=len(canonical_tags),
                           n_coarse=len(COARSE_CATEGORIES), beats_dim=beats_dim,
                           mel_frontend=mel_frontend).to(device)
    teacher.load_state_dict(model.state_dict())
    freeze_model(teacher)

    pseudo_overrides_global = {}

    stage1 = train_one_stage(
        model, None, full_dataset, tier_indices_global, val_full_gold,
        canonical_tags, raw_to_canonical, f2c, pseudo_overrides_global,
        median_duration, device, config,
        stage_name="STAGE 1: Gold+Silver | cached BEATs",
        include_bronze=False, use_mean_teacher=False,
        epochs=config["stage1_epochs"], lr=config["lr_stage1"],
        checkpoint_path="/kaggle/working/best_stage1.pt",
        wav_cache_dir=WAV_CACHE_DIRS, label_table=label_table)

    teacher.load_state_dict(model.state_dict())
    stage2 = train_one_stage(
        model, teacher, full_dataset, tier_indices_global, val_full_gold,
        canonical_tags, raw_to_canonical, f2c, pseudo_overrides_global,
        median_duration, device, config,
        stage_name="STAGE 2: + Bronze (mean-teacher)",
        include_bronze=True, use_mean_teacher=True,
        epochs=config["stage2_epochs"], lr=config["lr_stage2"],
        checkpoint_path="/kaggle/working/best_stage2.pt",
        wav_cache_dir=WAV_CACHE_DIRS, label_table=label_table)

    print("\n=== Generating pseudo-labels for Bronze ===")
    pseudo_overrides_global = generate_pseudo_labels(
        teacher, full_dataset, tier_indices_global["bronze"],
        config["sample_rate"], device,
        decode_params=stage2["tuned_decode_params"],
        beats_cache_dir=BEATS_CACHE_DIRS, wav_cache_dir=WAV_CACHE_DIRS,
        window_seconds=config["crop_seconds"])
    print(f"accepted pseudo-labels for {len(pseudo_overrides_global)}/"
          f"{len(tier_indices_global['bronze'])} Bronze clips "
          f"(keys are FULL-dataset indices)")

    stage3 = train_one_stage(
        model, None, full_dataset, tier_indices_global, val_full_gold,
        canonical_tags, raw_to_canonical, f2c, pseudo_overrides_global,
        median_duration, device, config,
        stage_name="STAGE 3: + pseudo-Bronze | head-only fine-tune",
        include_bronze=True, use_mean_teacher=False,
        epochs=config["stage3_epochs"], lr=config["lr_stage3"],
        checkpoint_path="/kaggle/working/best_stage3.pt",
        wav_cache_dir=WAV_CACHE_DIRS, label_table=label_table)

    print("\n=== TRAINING DONE ===")
    print(f"Stage 1 best combined: {stage1['best_combined']:.4f}")
    print(f"Stage 2 best combined: {stage2['best_combined']:.4f}")
    print(f"Stage 3 best combined: {stage3['best_combined']:.4f}")
    print("Final tuned decode params:", stage3["tuned_decode_params"])

    return {"model": model, "teacher": teacher, "canonical_tags": canonical_tags,
            "raw_to_canonical": raw_to_canonical, "fine_to_coarse_idx": f2c,
            "final_decode_params": stage3["tuned_decode_params"],
            "median_duration": median_duration}

In [ ]:
# =============================================================================
# CELL: BUILD BEATs CACHE (run once)
# =============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
full_dataset = load_from_disk(DATASET_PATH)
print("rows:", len(full_dataset))
beats_model, _ = load_beats_model(BEATS_CHECKPOINT, device)
# No packaged dataset to check anymore (see Setup) -- also_check_dirs=None.
# BEATS_CACHE_BUILD_DIR itself is always checked regardless (cache_dir is
# always in read_dirs), so any partial progress already on disk from an
# earlier run in this same session is still resumed, not recomputed.
build_beats_cache(full_dataset, beats_model, device, BEATS_CACHE_BUILD_DIR,
                  also_check_dirs=None)

In [ ]:
# =============================================================================
# CELL: Verify BEATs cache before training
#
# Exhaustive scan of BEATS_CACHE_BUILD_DIR (the only cache tier now): every
# file is checked for NaN/Inf and all-zero content, plus exhaustive index
# coverage against everything the 3-stage curriculum will touch. This is
# what would have caught the earlier all-zero corruption before it ever
# reached training -- run it after every fresh BUILD BEATs CACHE, before
# TRAIN.
# =============================================================================
import glob, time

_train_idx_lc, _val_idx_lc = split_train_val(
    full_dataset, val_fraction=CONFIG["val_fraction"], seed=CONFIG["seed"])
_val_quality_lc = full_dataset["annotationQuality"]
_val_gold_lc = [int(i) for i in _val_idx_lc if _val_quality_lc[i] == "verified_timestamps"]
_all_required_lc = sorted(set(int(i) for i in _train_idx_lc) | set(_val_gold_lc))

print("=" * 70)
print("STEP 1: index-coverage check against BEATS_CACHE_BUILD_DIR alone")
print("=" * 70)
print(f"Checking {len(_all_required_lc)} required indices against \n  {BEATS_CACHE_BUILD_DIR}")
_missing_local = [i for i in _all_required_lc
                  if not os.path.exists(os.path.join(BEATS_CACHE_BUILD_DIR, f"{i:06d}.npy"))]
if _missing_local:
    print(f"  INCOMPLETE: {len(_missing_local)} indices missing.")
    print(f"  first 10 missing: {_missing_local[:10]}")
    print(f"  (2 missing is expected/OK if they are exactly the known-corrupt "
          f"source clips [3561, 48940] -- anything else means the build above "
          f"did not finish; re-run the BUILD BEATs CACHE cell.)")
else:
    print(f"  Complete: all {len(_all_required_lc)} required indices present locally.")

print("\n" + "=" * 70)
print("STEP 2: exhaustive value scan (every file actually on disk)")
print("=" * 70)
_all_local_files = sorted(glob.glob(os.path.join(BEATS_CACHE_BUILD_DIR, "*.npy")))
print(f"found {len(_all_local_files)} .npy files -- scanning all of them")

_nan_bad_lc, _zero_bad_lc = [], []
_shape_counter_lc, _dtype_counter_lc = {}, {}
_t0 = time.time()
for _n, _path in enumerate(_all_local_files, 1):
    _arr = np.load(_path)
    _idx = int(os.path.basename(_path).replace(".npy", ""))
    _shape_counter_lc[_arr.shape] = _shape_counter_lc.get(_arr.shape, 0) + 1
    _dtype_counter_lc[str(_arr.dtype)] = _dtype_counter_lc.get(str(_arr.dtype), 0) + 1
    _arr32 = _arr.astype(np.float32, copy=False)
    if not np.isfinite(_arr32).all():
        _nan_bad_lc.append(_idx)
    elif not _arr.any():
        _zero_bad_lc.append(_idx)
    if _n % 10000 == 0:
        _rate = _n / (time.time() - _t0)
        _eta = (len(_all_local_files) - _n) / max(_rate, 1e-9)
        print(f"  [{_n:>6}/{len(_all_local_files)}]  {_rate:6.1f} files/s  "
              f"eta {_eta:5.1f}s  nan/inf={len(_nan_bad_lc)}  all-zero={len(_zero_bad_lc)}")

print(f"\ndone in {time.time() - _t0:.1f}s")
print(f"NaN/Inf files : {len(_nan_bad_lc)}  (first 10: {_nan_bad_lc[:10]})")
print(f"all-zero files: {len(_zero_bad_lc)}  (first 10: {_zero_bad_lc[:10]})")
print(f"shapes seen ({len(_shape_counter_lc)} unique): "
      f"{sorted(_shape_counter_lc.items(), key=lambda kv: -kv[1])[:5]}")
print(f"dtypes seen: {_dtype_counter_lc}")

print("\n" + "=" * 70)
if _missing_local or _nan_bad_lc or _zero_bad_lc:
    print(">>> NOT CLEAN -- do NOT zip yet. Fix the issues above first. <<<")
else:
    print(">>> Local BEATs cache verified: complete, all values finite and non-zero. <<<")
    print(">>> Safe to zip and upload as a new Kaggle Dataset. <<<")
print("=" * 70)


In [ ]:
# =============================================================================
# CELL: BUILD WAVEFORM CACHE (run once, AFTER the BEATs cache verification
# cell above reports clean)
#
# Decode+resample every clip ONCE instead of once per epoch. Stored as
# FLAC -- see build_waveform_cache's docstring for why, not raw .npy.
# Covers gold+silver+bronze (everything the 3-stage curriculum trains on)
# plus Gold-only val (the only subset train_one_stage ever actually
# evaluates on -- caching the rest of val would just waste disk and time).
# Anything not covered here (e.g. future pseudo-Bronze indices) self-heals
# via _load_waveform's fallback-and-cache behavior -- only their first
# access is ever slow.
# =============================================================================
_train_idx_wc, _val_idx_wc = split_train_val(
    full_dataset, val_fraction=CONFIG["val_fraction"], seed=CONFIG["seed"])
_tier_indices_wc = split_indices_by_tier(full_dataset, indices=_train_idx_wc)
_val_quality_wc = full_dataset["annotationQuality"]
_val_gold_wc = [int(i) for i in _val_idx_wc if _val_quality_wc[i] == "verified_timestamps"]

_all_wav_needed = sorted(
    set(int(i) for tier in _tier_indices_wc.values() for i in tier) | set(_val_gold_wc)
)
print(f"[wav-cache] {len(_all_wav_needed)} total indices need a waveform cache entry "
      f"(train: gold+silver+bronze, val: Gold-only)")
build_waveform_cache(
    full_dataset, _all_wav_needed, WAV_CACHE_BUILD_DIR,
    sample_rate=CONFIG["sample_rate"])


In [19]:
import numpy as np, glob, os, random

CACHE_DIR = BEATS_CACHE_DIR  # use your existing variable, not a retyped path
CLIP_BOUND = 6.0e4           # matches the clip in build_beats_cache

all_files = glob.glob(os.path.join(CACHE_DIR, "*.npy"))
print(f"total cached files found: {len(all_files)}")

random.seed(0)
sample_files = random.sample(all_files, min(500, len(all_files)))

global_min, global_max = np.inf, -np.inf
sum_, sumsq, n_vals = 0.0, 0.0, 0
n_files_touching_bound = 0
n_files_checked = 0
per_file_absmax = []

for path in sample_files:
    arr = np.load(path).astype(np.float32)   # upcast for safe stats even though stored as fp16
    if arr.size == 0 or not np.isfinite(arr).all():
        print(f"  !! non-finite values still present in {path}")
        continue
    n_files_checked += 1
    amax = float(np.abs(arr).max())
    per_file_absmax.append(amax)
    global_min = min(global_min, float(arr.min()))
    global_max = max(global_max, float(arr.max()))
    sum_ += float(arr.sum())
    sumsq += float((arr.astype(np.float64) ** 2).sum())
    n_vals += arr.size
    if amax >= CLIP_BOUND * 0.999:   # essentially pinned at the clip bound
        n_files_touching_bound += 1

mean = sum_ / n_vals
std = (sumsq / n_vals - mean ** 2) ** 0.5
per_file_absmax = np.array(per_file_absmax)

print(f"\nchecked {n_files_checked} files, {n_vals} total values")
print(f"global min / max : {global_min:.2f} / {global_max:.2f}")
print(f"mean / std        : {mean:.4f} / {std:.4f}")
print(f"files with max(|x|) >= 99.9% of clip bound ({CLIP_BOUND}): {n_files_touching_bound} / {n_files_checked}")
print(f"\nper-file max(|x|) percentiles:")
for p in (50, 75, 90, 95, 99, 100):
    print(f"  p{p}: {np.percentile(per_file_absmax, p):.2f}")
print(f"\nfiles with per-file max(|x|) > 100   : {(per_file_absmax > 100).sum()}")
print(f"files with per-file max(|x|) > 1000  : {(per_file_absmax > 1000).sum()}")
print(f"files with per-file max(|x|) > 10000 : {(per_file_absmax > 10000).sum()}")

total cached files found: 0


ZeroDivisionError: float division by zero

In [60]:
import glob

def _scan_cache_glob(cache_dir, log_every=5000):
    """Exhaustive scan over every .npy file that ACTUALLY EXISTS on disk --
    glob-based, not manifest-based, since the manifest undercounts if a
    prior run's files are still there from before a fix was applied.
    Flags NaN/Inf (shouldn't exist post-sanitize) AND exact all-zero
    content (the real signature of pre-fix garbage laundered by
    nan_to_num -- it isn't NaN anymore, so a NaN scan alone won't see it)."""
    paths = sorted(glob.glob(os.path.join(cache_dir, "*.npy")))
    print(f"[glob-scan] found {len(paths)} .npy files on disk")

    nan_bad, zero_bad = [], []
    for n_checked, path in enumerate(paths, 1):
        arr = np.load(path)
        idx = int(os.path.basename(path).replace(".npy", ""))
        if not np.isfinite(arr.astype(np.float32)).all():
            nan_bad.append(idx)
        elif not arr.any():          # exactly all zero
            zero_bad.append(idx)
        if n_checked % log_every == 0:
            print(f"  [glob-scan] {n_checked}/{len(paths)} checked, "
                  f"{len(nan_bad)} nan/inf, {len(zero_bad)} all-zero so far")

    print(f"[glob-scan] done. {len(nan_bad)} NaN/Inf, {len(zero_bad)} all-zero, "
          f"out of {len(paths)} total.")
    if zero_bad:
        print(f"  first 20 all-zero indices: {zero_bad[:20]}")
        print(f"  last 20 all-zero indices:  {zero_bad[-20:]}")
    return nan_bad, zero_bad

nan_bad, zero_bad = _scan_cache_glob(BEATS_CACHE_DIR)

[glob-scan] found 11391 .npy files on disk
  [glob-scan] 5000/11391 checked, 0 nan/inf, 0 all-zero so far
  [glob-scan] 10000/11391 checked, 0 nan/inf, 0 all-zero so far
[glob-scan] done. 0 NaN/Inf, 0 all-zero, out of 11391 total.


In [61]:
train_idx, val_idx = split_train_val(full_dataset,
                                      val_fraction=CONFIG["val_fraction"],
                                      seed=CONFIG["seed"])

def _has_cache(i, cache_dir=None):
    return resolve_cache_path(i, cache_dir if cache_dir is not None else BEATS_CACHE_DIRS) is not None

dropped_train = [i for i in train_idx if not _has_cache(i)]
dropped_val   = [i for i in val_idx   if not _has_cache(i)]
print(f"train rows missing cache: {len(dropped_train)} / {len(train_idx)}")
print(f"val rows missing cache:   {len(dropped_val)} / {len(val_idx)}")

val_quality = full_dataset["annotationQuality"]
val_full_gold = [int(i) for i in val_idx
                 if val_quality[i] == "verified_timestamps" and _has_cache(i)]
train_idx_clean = [i for i in train_idx if _has_cache(i)]

all_required = sorted(set(train_idx_clean) | set(val_full_gold))
_validate_cache_exhaustive(all_required, BEATS_CACHE_DIRS, label="pre-check")

train rows missing cache: 2 / 80166
val rows missing cache:   0 / 8281
  [pre-check] [cache-validate] all 81531 required indices present across ['/kaggle/input/datasets/ayanokoji2028/beats-cache-full-2/beats_cache_full', '/kaggle/working/beats_cache_full']


True

In [39]:
import torch
from torch.amp import autocast

sample = full_dataset[0]
wav_np, sr = _load_audio(sample["audio"])
wav = torch.from_numpy(np.asarray(wav_np, dtype=np.float32))
if wav.ndim > 1: wav = wav.mean(dim=-1)
if sr != 16000: wav = torchaudio.functional.resample(wav, sr, 16000)
wav = wav.to(device).unsqueeze(0)

with torch.no_grad():
    with autocast(device_type="cuda", enabled=True):
        feat_fp16, _, _ = beats_extract_temporal(beats_model, wav)
    print("WITH autocast (fp16): NaN?", torch.isnan(feat_fp16).any().item(),
          "Inf?", torch.isinf(feat_fp16).any().item(),
          "all-zero?", (feat_fp16 == 0).all().item())

    with autocast(device_type="cuda", enabled=False):
        feat_fp32, _, _ = beats_extract_temporal(beats_model, wav)
    print("WITHOUT autocast (fp32): NaN?", torch.isnan(feat_fp32).any().item(),
          "Inf?", torch.isinf(feat_fp32).any().item(),
          "min/max:", feat_fp32.min().item(), feat_fp32.max().item())

WITH autocast (fp16): NaN? True Inf? False all-zero? False
WITHOUT autocast (fp32): NaN? False Inf? False min/max: -1.5807677507400513 2.2861366271972656


In [20]:
import shutil
shutil.rmtree(BEATS_CACHE_DIR, ignore_errors=True)
os.makedirs(BEATS_CACHE_DIR, exist_ok=True)

In [ ]:
# =============================================================================
# CELL: TRAIN (assumes BEATS_CACHE_DIR is populated)
# =============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
full_dataset = load_from_disk(DATASET_PATH)
print("rows:", len(full_dataset))
results = main_three_stage(full_dataset, CONFIG, device)

device: cuda


Loading dataset from disk:   0%|          | 0/35 [00:00<?, ?it/s]

rows: 88447
Dropped 2 train + 0 val rows with no BEATs cache entry.
  train dropped (first 10): [3561, 48940]
train split: 80164, val split: 8281 (full-dataset indices)
tier sizes (global): {'gold': 7554, 'silver': 55482, 'bronze': 17128}
val (Gold-only): 1367 / 8281
  [main] [cache-validate] all 81531 required indices present in /kaggle/working/beats_cache_full
auxiliary fine classes: 97
median Gold duration (train): 0.487s
BEATs cfg: layers= 12  dim= 768  patch= 16


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


BEATs loaded and frozen.

=== STAGE 1: Gold+Silver | cached BEATs ===
  training subset size: 63036 full-dataset rows
  [STAGE 1: Gold+Silver | cached BEATs] [cache-validate] all 63036 required indices present in /kaggle/working/beats_cache_full
  [index-sanity] first 5 local->global mappings:
    local=0 full_idx=1 cache=/kaggle/working/beats_cache_full/000001.npy
    local=1 full_idx=29 cache=/kaggle/working/beats_cache_full/000029.npy
    local=2 full_idx=30 cache=/kaggle/working/beats_cache_full/000030.npy
    local=3 full_idx=86 cache=/kaggle/working/beats_cache_full/000086.npy
    local=4 full_idx=89 cache=/kaggle/working/beats_cache_full/000089.npy
  [cache-verify] checking 5/63036 samples (expected canonical_T=50)
    OK: local=41905 full_idx=52803 dur=10.62s T_cached=66 expected~66 ratio=1.00
    OK: local=7296 full_idx=36802 dur=5.11s T_cached=31 expected~32 ratio=0.97
    OK: local=1639 full_idx=3722 dur=9.67s T_cached=60 expected~60 ratio=1.00
    OK: local=48598 full_idx=6